<a href="https://colab.research.google.com/github/muzazr/favera-model/blob/main/05_candidate_v5_momtsim_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open V5 in Colab"/></a>

# 05 Candidate V5: Global Transaction Risk Ranker on MoMTSim V2

Notebook ini memindahkan pipeline Candidate V4 ke **MoMTSim V2** sebagai controlled challenger experiment.

## Tujuan

1. Menguji apakah feature contract deployable V4 tetap berguna pada distribusi V2.
2. Menggunakan temporal split baru berdasarkan rentang `step` V2.
3. Membuat calibration reference dan operating point baru khusus V2.
4. Mengevaluasi PR-AUC terhadap class-prior baseline, top-fraction metrics, temporal stability, dan cold-start.
5. Mengekspor artefak terpisah sebagai `global_ranker_v5_momtsim_v2`.

## Batas interpretasi

- Label `isFraud` adalah label fraud sintetis dari MoMTSim, **bukan label perjudian**.
- Model dipakai sebagai global extreme-risk ranker, bukan keputusan fraud final.
- Threshold classifier tidak dipakai otomatis sebagai threshold produk.
- Signal produksi notebook ini dibatasi menjadi:
  - percentile ≥ 0.99 → `STRONG_GLOBAL_SIGNAL`
  - selain itu → `NO_STRONG_GLOBAL_SIGNAL`
- Fitur saldo penerima tetap dikeluarkan dari feature contract produksi karena umumnya tidak tersedia dari e-statement.

## Catatan runtime

MoMTSim V2 berisi sekitar 4,2 juta baris. Gunakan Google Colab High-RAM bila runtime standar kehabisan memori.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!pip install -q \
    polars \
    pyarrow \
    pandas \
    numpy \
    scikit-learn \
    lightgbm \
    shap \
    matplotlib \
    joblib

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/favera/model/transaction-anomaly"
)

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "outputs"

for directory in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    MODEL_DIR,
    OUTPUT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

In [ ]:
import random
import joblib
import numpy as np
import pandas as pd
import polars as pl

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    precision_score,
    recall_score,
)
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# ============================================================
# 01.0 LOAD AND VALIDATE RAW MOMTSIM V2
# ============================================================

V2_RAW_FILENAME = "momtsim_v2.csv"
DATA_PATH = RAW_DATA_DIR / V2_RAW_FILENAME

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset MoMTSim V2 tidak ditemukan. "
        f"Expected path: {DATA_PATH}"
    )

EXPECTED_RAW_COLUMNS_V5 = [
    "step",
    "transactionType",
    "amount",
    "initiator",
    "oldBalInitiator",
    "newBalInitiator",
    "recipient",
    "oldBalRecipient",
    "newBalRecipient",
    "isFraud",
]

lazy_df = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=100_000,
    ignore_errors=False,
    low_memory=True,
)

raw_schema_v5 = lazy_df.collect_schema()
raw_columns_v5 = raw_schema_v5.names()

if raw_columns_v5 != EXPECTED_RAW_COLUMNS_V5:
    raise ValueError(
        "Skema raw MoMTSim V2 tidak sesuai.\n"
        f"Expected: {EXPECTED_RAW_COLUMNS_V5}\n"
        f"Actual: {raw_columns_v5}"
    )

row_count = (
    lazy_df
    .select(pl.len().alias("row_count"))
    .collect(engine="streaming")
    .item()
)

sample_df = lazy_df.head(5).collect()

print("Dataset path:", DATA_PATH)
print("Jumlah transaksi:", f"{row_count:,}")
print("Jumlah kolom:", len(raw_columns_v5))
print("Schema:")
print(raw_schema_v5)

display(sample_df)


## 2. Memahami Struktur Dataset

In [ ]:
from IPython.display import display, Markdown
import polars as pl

schema = lazy_df.collect_schema()
columns = schema.names()

schema_df = pl.DataFrame({
    "column": columns,
    "data_type": [str(schema[col]) for col in columns]
})

display(Markdown(
    f"Dataset memiliki **{row_count:,} baris** dan "
    f"**{len(columns)} kolom**."
))

display(schema_df)

In [ ]:
preview_df = lazy_df.head(10).collect()

display(preview_df)

EDA_SAMPLE_SIZE = 100_000

eda_df = lazy_df.head(EDA_SAMPLE_SIZE).collect(
    engine="streaming"
)

display(Markdown(
    f"Sampel EDA menggunakan **{eda_df.height:,} baris**."
))

In [ ]:
null_summary = (
    lazy_df
    .select([
        pl.col(col).null_count().alias(col)
        for col in columns
    ])
    .collect(engine="streaming")
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["null_count"]
    )
    .with_columns(
        (
            pl.col("null_count") / row_count * 100
        ).round(4).alias("null_percentage")
    )
    .sort("null_count", descending=True)
)

columns_with_null = null_summary.filter(
    pl.col("null_count") > 0
)

if columns_with_null.height == 0:
    display(Markdown("**Tidak ditemukan missing value.**"))
else:
    display(columns_with_null)

In [ ]:
sample_schema = eda_df.schema

numeric_cols = [
    col
    for col, dtype in sample_schema.items()
    if dtype.is_numeric()
]

string_cols = [
    col
    for col, dtype in sample_schema.items()
    if dtype == pl.String
]

print("Kolom numerik:", numeric_cols)
print("Kolom teks:", string_cols)



In [ ]:
if numeric_cols:
    numeric_summary = eda_df.select(
        numeric_cols
    ).describe()

    display(numeric_summary)
else:
    display(Markdown(
        "**Tidak ditemukan kolom numerik pada hasil parsing awal.**"
    ))

In [ ]:
if string_cols:
    unique_summary = pl.DataFrame({
        "column": string_cols,
        "unique_count_sample": [
            eda_df[col].n_unique()
            for col in string_cols
        ]
    }).sort(
        "unique_count_sample",
        descending=True
    )

    display(unique_summary)

In [ ]:
duplicate_count_sample = int(
    eda_df.is_duplicated().sum()
)

duplicate_percentage = (
    duplicate_count_sample / eda_df.height * 100
)

display(Markdown(
    f"""
**Duplikasi identik pada sampel:**
- Jumlah: **{duplicate_count_sample:,} baris**
- Persentase: **{duplicate_percentage:.4f}%**
"""
))

In [ ]:
target_keywords = [
    "fraud",
    "label",
    "class",
    "target",
    "anomaly"
]

time_keywords = [
    "date",
    "time",
    "timestamp",
    "step"
]

target_candidates = [
    col for col in columns
    if any(keyword in col.lower() for keyword in target_keywords)
]

time_candidates = [
    col for col in columns
    if any(keyword in col.lower() for keyword in time_keywords)
]

display(Markdown(
    f"""
### Kandidat Kolom Penting

**Kandidat label:** `{target_candidates}`

**Kandidat waktu:** `{time_candidates}`
"""
))

In [ ]:
for target_col in target_candidates:
    target_distribution = (
        lazy_df
        .group_by(target_col)
        .agg(
            pl.len().alias("count")
        )
        .with_columns(
            (
                pl.col("count") / row_count * 100
            ).round(4).alias("percentage")
        )
        .sort("count", descending=True)
        .collect(engine="streaming")
    )

    display(Markdown(
        f"### Distribusi `{target_col}`"
    ))
    display(target_distribution)

In [ ]:
# ============================================================
# 02.1 FULL DATASET AUDIT MOMTSIM V2
# ============================================================

dataset_overview_v5 = (
    lazy_df
    .select([
        pl.len().alias("rows"),
        pl.col("isFraud").sum().alias("fraud_count"),
        (
            pl.len() - pl.col("isFraud").sum()
        ).alias("normal_count"),
        pl.col("isFraud").mean().alias("fraud_rate"),
        pl.col("step").min().alias("minimum_step"),
        pl.col("step").max().alias("maximum_step"),
        pl.col("step").n_unique().alias("unique_steps"),
        pl.col("initiator").n_unique().alias("unique_initiators"),
        pl.col("recipient").n_unique().alias("unique_recipients"),
        pl.col("amount").median().alias("amount_median"),
        pl.col("amount").mean().alias("amount_mean"),
        pl.col("amount").quantile(0.99).alias("amount_p99"),
        pl.col("amount").max().alias("amount_max"),
    ])
    .collect(engine="streaming")
)

fraud_by_transaction_type_raw_v5 = (
    lazy_df
    .group_by("transactionType")
    .agg([
        pl.len().alias("transaction_count"),
        pl.col("isFraud").sum().alias("fraud_count"),
        (
            pl.len() - pl.col("isFraud").sum()
        ).alias("normal_count"),
        pl.col("isFraud").mean().alias("fraud_rate"),
    ])
    .sort("transaction_count", descending=True)
    .collect(engine="streaming")
)

display(dataset_overview_v5)
display(fraud_by_transaction_type_raw_v5)

non_transfer_fraud_raw_v5 = (
    lazy_df
    .filter(
        (pl.col("transactionType") != "TRANSFER")
        & (pl.col("isFraud") == 1)
    )
    .select(pl.len())
    .collect(engine="streaming")
    .item()
)

if non_transfer_fraud_raw_v5 > 0:
    raise ValueError(
        "Fraud ditemukan di luar TRANSFER. "
        "Scope global model harus dievaluasi ulang."
    )

print("Fraud di luar TRANSFER:", non_transfer_fraud_raw_v5)


In [ ]:
import re


def normalize_column_name(name: str) -> str:
    """Mengubah nama kolom menjadi format snake_case."""
    name = name.strip()
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name)
    return name.strip("_").lower()


rename_map = {
    col: normalize_column_name(col)
    for col in columns
}

clean_lf = lazy_df.rename(rename_map)

display(
    pl.DataFrame({
        "original_column": list(rename_map.keys()),
        "clean_column": list(rename_map.values())
    })
)

In [ ]:
clean_schema = clean_lf.collect_schema()

clean_string_cols = [
    col
    for col, dtype in clean_schema.items()
    if dtype == pl.String
]

NULL_TOKENS = [
    "",
    "na",
    "n/a",
    "null",
    "none",
    "-"
]

clean_lf = clean_lf.with_columns([
    pl.when(
        pl.col(col)
        .str.strip_chars()
        .str.to_lowercase()
        .is_in(NULL_TOKENS)
    )
    .then(None)
    .otherwise(
        pl.col(col).str.strip_chars()
    )
    .alias(col)
    for col in clean_string_cols
])

display(Markdown(
    f"Kolom teks yang dibersihkan: **{len(clean_string_cols)} kolom**."
))

In [ ]:
# ============================================================
# 02.2 SAVE CLEAN MOMTSIM V2
# ============================================================

STAGE_1_PATH = (
    PROCESSED_DATA_DIR
    / "momtsim_v2_stage1_clean.parquet"
)

clean_lf.sink_parquet(
    str(STAGE_1_PATH),
    compression="zstd",
)

display(
    Markdown(
        f"""
### Data Cleaning MoMTSim V2 Selesai

File tersimpan di:

`{STAGE_1_PATH}`
"""
    )
)

clean_lf = pl.scan_parquet(
    STAGE_1_PATH
)


In [ ]:
import polars as pl
from IPython.display import display, Markdown

# Kolom utama berdasarkan dataset kamu
TIME_COL = "step"
TARGET_COL = "is_fraud"

# Pastikan kolom wajib tersedia
required_columns = [
    "step",
    "transaction_type",
    "amount",
    "initiator",
    "old_bal_initiator",
    "new_bal_initiator",
    "recipient",
    "old_bal_recipient",
    "new_bal_recipient",
    "is_fraud",
]

available_columns = clean_lf.collect_schema().names()

missing_columns = [
    col for col in required_columns
    if col not in available_columns
]

if missing_columns:
    raise ValueError(
        f"Kolom tidak ditemukan: {missing_columns}"
    )

print("Semua kolom wajib tersedia.")

In [ ]:
# ============================================================
# 03.0 BASE FEATURE ENGINEERING FOR CANDIDATE V5
# ============================================================

feature_lf = (
    clean_lf
    .with_columns([
        # Tipe data dibuat konsisten
        pl.col("step").cast(pl.Int64),
        pl.col("amount").cast(pl.Float64),
        pl.col("initiator").cast(pl.Int64),
        pl.col("recipient").cast(pl.String),
        pl.col("is_fraud").cast(pl.Int8).alias("target"),

        pl.col("transaction_type")
        .cast(pl.String)
        .str.strip_chars()
        .str.to_uppercase(),
    ])
    .filter(
        pl.col("step").is_not_null() &
        pl.col("amount").is_not_null() &
        pl.col("target").is_not_null()
    )
    .with_columns([
        # Mengurangi efek nominal yang sangat ekstrem
        pl.col("amount")
        .clip(lower_bound=0)
        .log1p()
        .alias("amount_log1p"),

        # Perubahan saldo aktual
        (
            pl.col("new_bal_initiator") -
            pl.col("old_bal_initiator")
        ).alias("initiator_balance_change"),

        (
            pl.col("new_bal_recipient") -
            pl.col("old_bal_recipient")
        ).alias("recipient_balance_change"),

        # Rasio nominal terhadap saldo awal
        (
            pl.col("amount") /
            (pl.col("old_bal_initiator").abs() + 1)
        ).alias("amount_to_initiator_balance_ratio"),

        (
            pl.col("amount") /
            (pl.col("old_bal_recipient").abs() + 1)
        ).alias("amount_to_recipient_balance_ratio"),

        # Seberapa besar saldo pengirim berkurang
        (
            (
                pl.col("old_bal_initiator") -
                pl.col("new_bal_initiator")
            )
            / (pl.col("old_bal_initiator").abs() + 1)
        ).alias("initiator_balance_drop_ratio"),

        # Penanda kondisi saldo tertentu
        (
            pl.col("old_bal_initiator") <= 0
        ).cast(pl.Int8).alias("initiator_old_balance_non_positive"),

        (
            pl.col("new_bal_initiator") < 0
        ).cast(pl.Int8).alias("initiator_new_balance_negative"),

        (
            pl.col("old_bal_recipient") == 0
        ).cast(pl.Int8).alias("recipient_old_balance_zero"),
    ])
)

In [ ]:
feature_lf = feature_lf.with_columns([
    # Saldo pengirim yang secara sederhana diharapkan
    (
        pl.col("old_bal_initiator") -
        pl.col("amount")
    ).alias("expected_new_bal_initiator"),

    # Saldo penerima yang secara sederhana diharapkan
    (
        pl.col("old_bal_recipient") +
        pl.col("amount")
    ).alias("expected_new_bal_recipient"),
])

feature_lf = feature_lf.with_columns([
    # Selisih antara saldo aktual dan saldo yang diharapkan
    (
        pl.col("new_bal_initiator") -
        pl.col("expected_new_bal_initiator")
    ).abs().alias("initiator_balance_error"),

    (
        pl.col("new_bal_recipient") -
        pl.col("expected_new_bal_recipient")
    ).abs().alias("recipient_balance_error"),
])

In [ ]:
import numpy as np
import polars as pl

# ============================================================
# FITUR WAKTU BERDASARKAN STEP
# 1 step MoMTSim = 1 jam
# ============================================================

feature_lf = feature_lf.with_columns([
    # Jam relatif dalam siklus 24 jam.
    # Ini belum tentu menunjukkan jam kalender sebenarnya
    # karena titik awal simulasi belum dikonfirmasi.
    (
        pl.col("step") % 24
    )
    .cast(pl.Int8)
    .alias("hour_of_day"),

    # Hari relatif sejak simulasi dimulai.
    (
        pl.col("step") // 24
    )
    .cast(pl.Int16)
    .alias("day_index"),
])

feature_lf = feature_lf.with_columns([
    # Fitur eksperimental.
    # Tidak akan dimasukkan ke global model V3 dahulu.
    (
        (pl.col("hour_of_day") >= 22)
        |
        (pl.col("hour_of_day") < 5)
    )
    .cast(pl.Int8)
    .alias("is_night"),

    # Representasi siklik.
    # Jam 23 dibuat dekat dengan jam 0.
    (
        2
        * np.pi
        * pl.col("hour_of_day")
        / 24
    )
    .sin()
    .alias("hour_sin"),

    (
        2
        * np.pi
        * pl.col("hour_of_day")
        / 24
    )
    .cos()
    .alias("hour_cos"),
])

# ============================================================
# PEMERIKSAAN STEP DATASET
# ============================================================

step_summary = (
    feature_lf
    .select([
        pl.col("step")
        .min()
        .alias("minimum_step"),

        pl.col("step")
        .max()
        .alias("maximum_step"),

        pl.col("step")
        .n_unique()
        .alias("unique_steps"),

        pl.col("hour_of_day")
        .min()
        .alias("minimum_relative_hour"),

        pl.col("hour_of_day")
        .max()
        .alias("maximum_relative_hour"),

        pl.col("day_index")
        .max()
        .alias("maximum_day_index"),
    ])
    .collect(
        engine="streaming"
    )
)

display(step_summary)

In [ ]:
# ============================================================
# 00.1 CONFIGURATION AND EXPERIMENT SETUP V5
# ============================================================

import gc
import json
import random
import joblib
import numpy as np
import pandas as pd
import polars as pl

from pathlib import Path
from IPython.display import display

from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
)

from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

EXPERIMENT_VERSION = "candidate_v5_transfer_global"
DATASET_VERSION = "momtsim_v2"
BASE_FEATURE_VERSION = "candidate_v3"
INTERACTION_FEATURE_VERSION = "candidate_v4_interactions_reused"
PREVIOUS_GLOBAL_MODEL = "global_ranker_v4_momtsim_v1"

# ============================================================
# TARGET SEMENTARA GLOBAL SUPERVISED MODEL
#
# Nilai absolut dipertahankan untuk continuity. Interpretasi
# V2 wajib disertai fraud-rate baseline dan PR-AUC lift.
# ============================================================

TARGET_PR_AUC = 0.50
TARGET_PRECISION = 0.50
TARGET_RECALL = 0.70
TARGET_F1 = 0.58
TARGET_FPR = 0.20

# ============================================================
# TARGET FINAL SETELAH SCORE FUSION
# ============================================================

APPLICATION_TARGETS = {
    "pr_auc": 0.60,
    "precision": 0.65,
    "recall": 0.80,
    "f1": 0.70,
    "false_positive_rate": 0.10,
    "false_alerts_per_user_week": 2.0,
}

print("Experiment:", EXPERIMENT_VERSION)
print("Dataset:", DATASET_VERSION)
print("Base features:", BASE_FEATURE_VERSION)
print("Interaction features:", INTERACTION_FEATURE_VERSION)
print("Previous global model:", PREVIOUS_GLOBAL_MODEL)
print("Seed:", SEED)


In [ ]:
# ============================================================
# 00.1A LOAD V4 MOMTSIM V1 BASELINE FOR COMPARISON
# ============================================================

V4_V1_ARTIFACT_DIR = (
    Path(MODEL_DIR)
    / "global_ranker_v4_momtsim_v1"
)

V4_V1_EVALUATION_PATH = (
    V4_V1_ARTIFACT_DIR
    / "evaluation_metrics_v4.json"
)

V4_V1_BASELINE_TEST_METRICS = None

if V4_V1_EVALUATION_PATH.exists():
    with open(
        V4_V1_EVALUATION_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        raw_v4_metrics = json.load(file)

    baseline_metric_names = [
        "fraud_rate",
        "pr_auc",
        "precision",
        "precision_lift",
        "recall",
        "f1",
        "false_positive_rate",
        "alert_rate",
    ]

    V4_V1_BASELINE_TEST_METRICS = {
        metric: raw_v4_metrics.get(metric)
        for metric in baseline_metric_names
    }

    print(
        "Baseline V4/V1 berhasil dimuat:",
        V4_V1_EVALUATION_PATH,
    )

    display(
        pd.DataFrame([
            {
                "model": "Candidate V4 / MoMTSim V1",
                **V4_V1_BASELINE_TEST_METRICS,
            }
        ])
    )
else:
    print(
        "WARNING: evaluation_metrics_v4.json tidak ditemukan. "
        "Eksperimen V5 tetap dapat berjalan tanpa row pembanding V4."
    )


In [ ]:
# ============================================================
# 00.2 OPERATING POINT TABLE UTILITY
# ============================================================
def build_operating_point_table(
    y_true,
    probabilities,
):
    """
    Membuat seluruh operating point berdasarkan kemungkinan threshold.
    """
    y_true = np.asarray(
        y_true,
        dtype=np.int8,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    if len(y_true) != len(probabilities):
        raise ValueError(
            "Jumlah target dan probabilitas tidak sama."
        )

    # Skor tertinggi diletakkan lebih dahulu.
    order = np.argsort(
        -probabilities,
        kind="mergesort",
    )

    sorted_target = y_true[order]
    sorted_probability = probabilities[order]

    true_positive = np.cumsum(
        sorted_target
    )

    false_positive = np.cumsum(
        1 - sorted_target
    )

    total_positive = int(
        sorted_target.sum()
    )

    total_negative = (
        len(sorted_target)
        - total_positive
    )

    false_negative = (
        total_positive
        - true_positive
    )

    true_negative = (
        total_negative
        - false_positive
    )

    precision = (
        true_positive /
        np.maximum(
            true_positive + false_positive,
            1,
        )
    )

    recall = (
        true_positive /
        max(total_positive, 1)
    )

    false_positive_rate = (
        false_positive /
        max(total_negative, 1)
    )

    f1 = (
        2 * precision * recall /
        np.maximum(
            precision + recall,
            1e-12,
        )
    )

    # Ambil posisi terakhir untuk probabilitas yang sama.
    distinct_indices = np.r_[
        np.flatnonzero(
            sorted_probability[:-1]
            != sorted_probability[1:]
        ),
        len(sorted_probability) - 1,
    ]

    return pd.DataFrame({
        "threshold": (
            sorted_probability[
                distinct_indices
            ]
        ),
        "precision": (
            precision[
                distinct_indices
            ]
        ),
        "recall": (
            recall[
                distinct_indices
            ]
        ),
        "f1": (
            f1[
                distinct_indices
            ]
        ),
        "false_positive_rate": (
            false_positive_rate[
                distinct_indices
            ]
        ),
        "true_positive": (
            true_positive[
                distinct_indices
            ]
        ),
        "false_positive": (
            false_positive[
                distinct_indices
            ]
        ),
        "false_negative": (
            false_negative[
                distinct_indices
            ]
        ),
        "true_negative": (
            true_negative[
                distinct_indices
            ]
        ),
    })

# Histori Pengirim

In [ ]:
required_history_columns = [
    "step",
    "initiator",
    "recipient",
    "amount",
]

available_feature_columns = (
    feature_lf
    .collect_schema()
    .names()
)

missing_history_columns = [
    column
    for column in required_history_columns
    if column not in available_feature_columns
]

if missing_history_columns:
    raise ValueError(
        "Kolom histori tidak ditemukan: "
        f"{missing_history_columns}"
    )

# Materialisasi hanya empat kolom yang diperlukan.
# Ini mencegah seluruh feature_lf masuk RAM sekaligus.
base_history_df = (
    feature_lf
    .select(
        required_history_columns
    )
    .collect(
        engine="streaming"
    )
)

print(
    "History source shape:",
    base_history_df.shape,
)

initiator_step_df = (
    base_history_df
    .group_by([
        "initiator",
        "step",
    ])
    .agg([
        pl.len()
        .alias(
            "initiator_tx_count_step"
        ),

        pl.col("amount")
        .sum()
        .alias(
            "initiator_amount_sum_step"
        ),

        pl.col("amount")
        .pow(2)
        .sum()
        .alias(
            "initiator_amount_sq_sum_step"
        ),

        pl.col("amount")
        .max()
        .alias(
            "initiator_amount_max_step"
        ),
    ])
    .sort([
        "initiator",
        "step",
    ])
)

print(
    "Initiator-step rows:",
    initiator_step_df.height,
)

In [ ]:
initiator_history_df = (
    initiator_step_df
    .with_columns([
        (
            pl.col(
                "initiator_tx_count_step"
            )
            .cum_sum()
            .over(
                "initiator",
                order_by="step",
            )
            -
            pl.col(
                "initiator_tx_count_step"
            )
        )
        .alias(
            "initiator_tx_count_prior"
        ),

        (
            pl.col(
                "initiator_amount_sum_step"
            )
            .cum_sum()
            .over(
                "initiator",
                order_by="step",
            )
            -
            pl.col(
                "initiator_amount_sum_step"
            )
        )
        .alias(
            "initiator_amount_sum_prior"
        ),

        (
            pl.col(
                "initiator_amount_sq_sum_step"
            )
            .cum_sum()
            .over(
                "initiator",
                order_by="step",
            )
            -
            pl.col(
                "initiator_amount_sq_sum_step"
            )
        )
        .alias(
            "initiator_amount_sq_sum_prior"
        ),

        pl.col(
            "initiator_amount_max_step"
        )
        .cum_max()
        .shift(1)
        .over(
            "initiator",
            order_by="step",
        )
        .alias(
            "initiator_amount_max_prior"
        ),

        pl.col("step")
        .shift(1)
        .over(
            "initiator",
            order_by="step",
        )
        .alias(
            "initiator_previous_step"
        ),
    ])
)

In [ ]:
initiator_6h_df = (
    initiator_step_df
    .select([
        "initiator",
        "step",
        "initiator_tx_count_step",
        "initiator_amount_sum_step",
    ])
    .sort([
        "initiator",
        "step",
    ])
    .rolling(
        index_column="step",
        period="6i",
        closed="left",
        group_by="initiator",
    )
    .agg([
        pl.col(
            "initiator_tx_count_step"
        )
        .sum()
        .alias(
            "initiator_tx_count_6h"
        ),

        pl.col(
            "initiator_amount_sum_step"
        )
        .sum()
        .alias(
            "initiator_amount_sum_6h"
        ),
    ])
)

In [ ]:
initiator_24h_df = (
    initiator_step_df
    .select([
        "initiator",
        "step",
        "initiator_tx_count_step",
        "initiator_amount_sum_step",
    ])
    .sort([
        "initiator",
        "step",
    ])
    .rolling(
        index_column="step",
        period="24i",
        closed="left",
        group_by="initiator",
    )
    .agg([
        pl.col(
            "initiator_tx_count_step"
        )
        .sum()
        .alias(
            "initiator_tx_count_24h"
        ),

        pl.col(
            "initiator_amount_sum_step"
        )
        .sum()
        .alias(
            "initiator_amount_sum_24h"
        ),
    ])
)

In [ ]:
initiator_history_df = (
    initiator_history_df
    .join(
        initiator_6h_df,
        on=[
            "initiator",
            "step",
        ],
        how="left",
    )
    .join(
        initiator_24h_df,
        on=[
            "initiator",
            "step",
        ],
        how="left",
    )
)

# Histori Penerima

In [ ]:
recipient_step_df = (
    base_history_df
    .group_by([
        "recipient",
        "step",
    ])
    .agg([
        pl.len()
        .alias(
            "recipient_tx_count_step"
        ),
    ])
    .sort([
        "recipient",
        "step",
    ])
)

In [ ]:
recipient_history_df = (
    recipient_step_df
    .with_columns([
        (
            pl.col(
                "recipient_tx_count_step"
            )
            .cum_sum()
            .over(
                "recipient",
                order_by="step",
            )
            -
            pl.col(
                "recipient_tx_count_step"
            )
        )
        .alias(
            "recipient_tx_count_prior"
        ),
    ])
    .select([
        "recipient",
        "step",
        "recipient_tx_count_prior",
    ])
)

# Histori Pasangan pengirim dan Penerima

In [ ]:
pair_step_df = (
    base_history_df
    .group_by([
        "initiator",
        "recipient",
        "step",
    ])
    .agg([
        pl.len()
        .alias(
            "pair_tx_count_step"
        ),
    ])
    .sort([
        "initiator",
        "recipient",
        "step",
    ])
)

pair_history_df = (
    pair_step_df
    .with_columns([
        (
            pl.col(
                "pair_tx_count_step"
            )
            .cum_sum()
            .over(
                [
                    "initiator",
                    "recipient",
                ],
                order_by="step",
            )
            -
            pl.col(
                "pair_tx_count_step"
            )
        )
        .alias(
            "pair_tx_count_prior"
        ),

        pl.col("step")
        .shift(1)
        .over(
            [
                "initiator",
                "recipient",
            ],
            order_by="step",
        )
        .alias(
            "pair_previous_step"
        ),
    ])
    .select([
        "initiator",
        "recipient",
        "step",
        "pair_tx_count_prior",
        "pair_previous_step",
    ])
)

In [ ]:
behavioral_feature_lf = (
    feature_lf
    .join(
        initiator_history_df.lazy(),
        on=[
            "initiator",
            "step",
        ],
        how="left",
    )
    .join(
        recipient_history_df.lazy(),
        on=[
            "recipient",
            "step",
        ],
        how="left",
    )
    .join(
        pair_history_df.lazy(),
        on=[
            "initiator",
            "recipient",
            "step",
        ],
        how="left",
    )
)

# Pengisian Null Value

In [ ]:
history_zero_columns = [
    "initiator_tx_count_prior",
    "initiator_amount_sum_prior",
    "initiator_amount_sq_sum_prior",
    "initiator_amount_max_prior",
    "initiator_tx_count_6h",
    "initiator_amount_sum_6h",
    "initiator_tx_count_24h",
    "initiator_amount_sum_24h",
    "recipient_tx_count_prior",
    "pair_tx_count_prior",
]

behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        pl.col(column)
        .fill_null(0)
        .alias(column)
        for column in history_zero_columns
    ])
)

# Fitur Behavioral V2

In [ ]:
behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        (
            pl.col(
                "initiator_tx_count_prior"
            ) > 0
        )
        .cast(pl.Int8)
        .alias(
            "has_initiator_history"
        ),

        (
            pl.col(
                "pair_tx_count_prior"
            ) == 0
        )
        .cast(pl.Int8)
        .alias(
            "is_new_recipient"
        ),

        pl.when(
            pl.col(
                "initiator_tx_count_prior"
            ) > 0
        )
        .then(
            pl.col(
                "initiator_amount_sum_prior"
            )
            /
            pl.col(
                "initiator_tx_count_prior"
            )
        )
        .otherwise(0)
        .alias(
            "initiator_amount_mean_prior"
        ),
    ])
)

behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        pl.when(
            pl.col(
                "initiator_tx_count_prior"
            ) > 1
        )
        .then(
            (
                pl.col(
                    "initiator_amount_sq_sum_prior"
                )
                /
                pl.col(
                    "initiator_tx_count_prior"
                )
            )
            -
            pl.col(
                "initiator_amount_mean_prior"
            ).pow(2)
        )
        .otherwise(0)
        .clip(lower_bound=0)
        .alias(
            "initiator_amount_variance_prior"
        ),
    ])
    .with_columns([
        pl.col(
            "initiator_amount_variance_prior"
        )
        .sqrt()
        .alias(
            "initiator_amount_std_prior"
        ),
    ])
)

behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        pl.when(
            pl.col(
                "initiator_tx_count_prior"
            ) > 0
        )
        .then(
            pl.col("amount")
            /
            (
                pl.col(
                    "initiator_amount_mean_prior"
                ).abs()
                + 1
            )
        )
        .otherwise(0)
        .alias(
            "amount_vs_initiator_mean"
        ),

        pl.when(
            pl.col(
                "initiator_tx_count_prior"
            ) > 1
        )
        .then(
            (
                pl.col("amount")
                -
                pl.col(
                    "initiator_amount_mean_prior"
                )
            )
            /
            (
                pl.col(
                    "initiator_amount_std_prior"
                )
                + 1
            )
        )
        .otherwise(0)
        .alias(
            "amount_zscore_initiator"
        ),

        pl.when(
            pl.col(
                "initiator_amount_max_prior"
            ) > 0
        )
        .then(
            pl.col("amount")
            /
            (
                pl.col(
                    "initiator_amount_max_prior"
                )
                + 1
            )
        )
        .otherwise(0)
        .alias(
            "amount_vs_initiator_max"
        ),

        pl.when(
            pl.col(
                "initiator_previous_step"
            ).is_not_null()
        )
        .then(
            pl.col("step")
            -
            pl.col(
                "initiator_previous_step"
            )
        )
        .otherwise(-1)
        .alias(
            "steps_since_initiator_transaction"
        ),

        pl.when(
            pl.col(
                "pair_previous_step"
            ).is_not_null()
        )
        .then(
            pl.col("step")
            -
            pl.col(
                "pair_previous_step"
            )
        )
        .otherwise(-1)
        .alias(
            "steps_since_pair_transaction"
        ),
    ])
)

# Fitur V3

In [ ]:
# ============================================================
# 03.5 CANDIDATE V3 DERIVED BEHAVIORAL FEATURES
# ============================================================

behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        (
            pl.col("amount_log1p")
            -
            pl.col("initiator_amount_mean_prior")
            .clip(lower_bound=0)
            .log1p()
        )
        .alias("amount_log_ratio_to_mean"),

        (
            pl.col("initiator_tx_count_6h")
            /
            (
                pl.col("initiator_tx_count_24h") / 4
                + 1
            )
        )
        .alias("tx_velocity_6h_vs_24h"),

        pl.when(
            pl.col("initiator_tx_count_6h") > 0
        )
        .then(
            pl.col("amount")
            /
            (
                pl.col("initiator_amount_sum_6h")
                /
                pl.col("initiator_tx_count_6h")
                + 1
            )
        )
        .otherwise(0)
        .alias("amount_vs_recent_mean_6h"),

        pl.when(
            pl.col("initiator_tx_count_24h") > 0
        )
        .then(
            pl.col("amount")
            /
            (
                pl.col("initiator_amount_sum_24h")
                /
                pl.col("initiator_tx_count_24h")
                + 1
            )
        )
        .otherwise(0)
        .alias("amount_vs_recent_mean_24h"),

        pl.col("recipient_tx_count_prior")
        .cast(pl.Float64)
        .log1p()
        .alias("recipient_history_log1p"),

        pl.col("pair_tx_count_prior")
        .cast(pl.Float64)
        .log1p()
        .alias("pair_history_log1p"),

        (
            (
                pl.col("steps_since_initiator_transaction")
                >= 0
            )
            &
            (
                pl.col("steps_since_initiator_transaction")
                <= 1
            )
        )
        .cast(pl.Int8)
        .alias("is_rapid_initiator_repeat"),

        (
            (
                pl.col("steps_since_pair_transaction")
                >= 0
            )
            &
            (
                pl.col("steps_since_pair_transaction")
                <= 1
            )
        )
        .cast(pl.Int8)
        .alias("is_rapid_pair_repeat"),

        (
            pl.col("new_bal_initiator")
            /
            (
                pl.col("old_bal_initiator").abs()
                + 1
            )
        )
        .alias("remaining_balance_ratio"),

        (
            pl.col("initiator_tx_count_prior") < 5
        )
        .cast(pl.Int8)
        .alias("is_cold_start"),
    ])
)

In [ ]:
# ============================================================
# 03.6 CANDIDATE V5 USER HISTORY COVERAGE AUDIT
# ============================================================

initiator_lifetime_count_v5 = (
    base_history_df
    .group_by("initiator")
    .agg(
        pl.len().alias("transaction_count")
    )
)

initiator_count_summary_v5 = (
    initiator_lifetime_count_v5
    .select([
        pl.len().alias("unique_initiators"),
        pl.col("transaction_count").mean().alias("mean_transactions"),
        pl.col("transaction_count").median().alias("median_transactions"),
        pl.col("transaction_count").quantile(0.75).alias("p75_transactions"),
        pl.col("transaction_count").quantile(0.90).alias("p90_transactions"),
        pl.col("transaction_count").quantile(0.95).alias("p95_transactions"),
        pl.col("transaction_count").quantile(0.99).alias("p99_transactions"),
        pl.col("transaction_count").max().alias("maximum_transactions"),
    ])
)

initiator_history_segments_v5 = (
    initiator_lifetime_count_v5
    .with_columns(
        pl.when(pl.col("transaction_count") == 1)
        .then(pl.lit("1_transaction"))
        .when(pl.col("transaction_count") < 5)
        .then(pl.lit("2_to_4_transactions"))
        .otherwise(pl.lit("5_or_more_transactions"))
        .alias("history_segment")
    )
    .group_by("history_segment")
    .agg(pl.len().alias("initiators"))
    .with_columns(
        (
            pl.col("initiators")
            / pl.col("initiators").sum()
        ).alias("initiator_share")
    )
    .sort("initiators", descending=True)
)

row_level_history_summary_v5 = (
    behavioral_feature_lf
    .select([
        pl.len().alias("rows"),
        pl.col("has_initiator_history").mean().alias("has_history_rate"),
        pl.col("is_cold_start").mean().alias("cold_start_rate"),
        pl.col("initiator_tx_count_prior").mean().alias("mean_prior_transactions"),
        pl.col("initiator_tx_count_prior").median().alias("median_prior_transactions"),
    ])
    .collect(engine="streaming")
)

display(initiator_count_summary_v5)
display(initiator_history_segments_v5)
display(row_level_history_summary_v5)


In [ ]:
behavioral_preview = (
    behavioral_feature_lf
    .select([
        "step",
        "initiator",
        "recipient",
        "amount",
        "initiator_tx_count_prior",
        "initiator_amount_mean_prior",
        "amount_vs_initiator_mean",
        "initiator_tx_count_6h",
        "initiator_tx_count_24h",
        "recipient_tx_count_prior",
        "pair_tx_count_prior",
        "is_new_recipient",
        "target",
    ])
    .head(10)
    .collect()
)

display(behavioral_preview)

In [ ]:
behavioral_row_count = (
    behavioral_feature_lf
    .select(
        pl.len()
    )
    .collect(
        engine="streaming"
    )
    .item()
)

print(
    "Rows before behavioral features:",
    row_count,
)

print(
    "Rows after behavioral features:",
    behavioral_row_count,
)

if behavioral_row_count != row_count:
    raise ValueError(
        "Jumlah baris berubah setelah join behavioral features."
    )

In [ ]:
# ============================================================
# 04.0 CANDIDATE V5 LABEL AUDIT BY TRANSACTION TYPE
# ============================================================

fraud_by_transaction_type_v5 = (
    behavioral_feature_lf
    .group_by("transaction_type")
    .agg([
        pl.len().alias("transaction_count"),
        pl.col("target").sum().alias("fraud_count"),
        (
            pl.len() - pl.col("target").sum()
        ).alias("normal_count"),
        pl.col("target").mean().alias("fraud_rate"),
    ])
    .sort("transaction_count", descending=True)
    .collect(engine="streaming")
)

display(fraud_by_transaction_type_v5)

non_transfer_fraud_count_v5 = (
    behavioral_feature_lf
    .filter(
        (pl.col("transaction_type") != "TRANSFER")
        & (pl.col("target") == 1)
    )
    .select(pl.len())
    .collect(engine="streaming")
    .item()
)

transfer_class_summary_v5 = (
    behavioral_feature_lf
    .filter(pl.col("transaction_type") == "TRANSFER")
    .select([
        pl.len().alias("transfer_rows"),
        pl.col("target").sum().alias("fraud_count"),
        (
            pl.len() - pl.col("target").sum()
        ).alias("normal_count"),
        pl.col("target").mean().alias("fraud_rate"),
    ])
    .collect(engine="streaming")
)

display(transfer_class_summary_v5)

if non_transfer_fraud_count_v5 > 0:
    raise ValueError(
        "Ditemukan fraud di luar TRANSFER. "
        "Scope global model harus ditinjau ulang."
    )

if int(transfer_class_summary_v5["normal_count"][0]) == 0:
    raise ValueError("TRANSFER V2 tidak memiliki kelas normal.")

if int(transfer_class_summary_v5["fraud_count"][0]) == 0:
    raise ValueError("TRANSFER V2 tidak memiliki kelas fraud.")

print("Fraud di luar TRANSFER:", non_transfer_fraud_count_v5)


In [ ]:
# ============================================================
# 06.0 TEMPORAL SPLIT CANDIDATE V5 ON MOMTSIM V2
# Train 70%
# Validation 10%
# Calibration 10%
# Test 10%
# ============================================================

unique_steps_v3 = (
    behavioral_feature_lf
    .select("step")
    .unique()
    .sort("step")
    .collect(
        engine="streaming"
    )
    .get_column("step")
)

number_of_steps_v3 = len(
    unique_steps_v3
)

if number_of_steps_v3 < 40:
    raise ValueError(
        "Jumlah step terlalu sedikit "
        "untuk empat temporal split."
    )

train_step_index_v3 = (
    int(
        number_of_steps_v3 * 0.70
    )
    - 1
)

validation_step_index_v3 = (
    int(
        number_of_steps_v3 * 0.80
    )
    - 1
)

calibration_step_index_v3 = (
    int(
        number_of_steps_v3 * 0.90
    )
    - 1
)

train_end_step_v3 = (
    unique_steps_v3[
        train_step_index_v3
    ]
)

validation_end_step_v3 = (
    unique_steps_v3[
        validation_step_index_v3
    ]
)

calibration_end_step_v3 = (
    unique_steps_v3[
        calibration_step_index_v3
    ]
)

print(
    "Jumlah step:",
    number_of_steps_v3,
)

print(
    "Train:",
    unique_steps_v3[0],
    "sampai",
    train_end_step_v3,
)

print(
    "Validation:",
    train_end_step_v3 + 1,
    "sampai",
    validation_end_step_v3,
)

print(
    "Calibration:",
    validation_end_step_v3 + 1,
    "sampai",
    calibration_end_step_v3,
)

print(
    "Test:",
    calibration_end_step_v3 + 1,
    "sampai",
    unique_steps_v3[-1],
)

In [ ]:
# ============================================================
# 06.1 GLOBAL MODEL V5
# HANYA TRANSAKSI TRANSFER
# ============================================================

global_model_filter_v3 = (
    pl.col("transaction_type")
    == "TRANSFER"
)

train_v3_lf = (
    behavioral_feature_lf
    .filter(
        global_model_filter_v3
        &
        (
            pl.col("step")
            <= train_end_step_v3
        )
    )
)

validation_v3_lf = (
    behavioral_feature_lf
    .filter(
        global_model_filter_v3
        &
        (
            pl.col("step")
            > train_end_step_v3
        )
        &
        (
            pl.col("step")
            <= validation_end_step_v3
        )
    )
)

calibration_v3_lf = (
    behavioral_feature_lf
    .filter(
        global_model_filter_v3
        &
        (
            pl.col("step")
            > validation_end_step_v3
        )
        &
        (
            pl.col("step")
            <= calibration_end_step_v3
        )
    )
)

test_v3_lf = (
    behavioral_feature_lf
    .filter(
        global_model_filter_v3
        &
        (
            pl.col("step")
            > calibration_end_step_v3
        )
    )
)

In [ ]:
# ============================================================
# 06.2 TEMPORAL SPLIT SUMMARY V5
# ============================================================

def summarize_split_v3(
    split_name,
    split_lf,
):
    return (
        split_lf
        .select([
            pl.lit(
                split_name
            )
            .alias("split"),

            pl.len()
            .alias("row_count"),

            pl.col("step")
            .min()
            .alias("step_start"),

            pl.col("step")
            .max()
            .alias("step_end"),

            pl.col("target")
            .sum()
            .alias("fraud_count"),

            (
                pl.len()
                - pl.col("target").sum()
            )
            .alias("normal_count"),

            pl.col("target")
            .mean()
            .alias("fraud_rate"),

            pl.col("initiator")
            .n_unique()
            .alias("unique_initiators"),

            pl.col("recipient")
            .n_unique()
            .alias("unique_recipients"),
        ])
        .collect(
            engine="streaming"
        )
    )


split_summary_v3 = pl.concat([
    summarize_split_v3(
        "train",
        train_v3_lf,
    ),
    summarize_split_v3(
        "validation",
        validation_v3_lf,
    ),
    summarize_split_v3(
        "calibration",
        calibration_v3_lf,
    ),
    summarize_split_v3(
        "test",
        test_v3_lf,
    ),
])

display(
    split_summary_v3
)

for split_row in split_summary_v3.iter_rows(
    named=True
):
    if split_row["row_count"] == 0:
        raise ValueError(
            f"Split {split_row['split']} kosong."
        )

    if split_row["fraud_count"] == 0:
        raise ValueError(
            f"Split {split_row['split']} "
            "tidak memiliki fraud."
        )

    if split_row["normal_count"] == 0:
        raise ValueError(
            f"Split {split_row['split']} "
            "tidak memiliki transaksi normal."
        )

In [ ]:
# ============================================================
# 06.3 CANDIDATE V5 SPLIT BEHAVIOR AND COLD-START AUDIT
# ============================================================

def summarize_split_behavior_v5(
    split_name,
    split_lf,
):
    return (
        split_lf
        .select([
            pl.lit(split_name).alias("split"),
            pl.len().alias("rows"),
            pl.col("target").mean().alias("fraud_rate"),
            pl.col("is_cold_start").mean().alias("cold_start_rate"),
            pl.col("has_initiator_history").mean().alias("has_history_rate"),
            pl.col("initiator_tx_count_prior").mean().alias("mean_prior_transactions"),
            pl.col("initiator_tx_count_prior").median().alias("median_prior_transactions"),
            pl.col("is_new_recipient").mean().alias("new_recipient_rate"),
        ])
        .collect(engine="streaming")
    )

split_behavior_summary_v5 = pl.concat([
    summarize_split_behavior_v5("train", train_v3_lf),
    summarize_split_behavior_v5("validation", validation_v3_lf),
    summarize_split_behavior_v5("calibration", calibration_v3_lf),
    summarize_split_behavior_v5("test", test_v3_lf),
])

display(split_behavior_summary_v5)


In [ ]:
# ============================================================
# 06.4 PROCESSED SPLIT PATHS CANDIDATE V5 MOMTSIM V2
# ============================================================

TRAIN_V3_PATH = (
    PROCESSED_DATA_DIR
    / "momtsim_v2_train_features_candidate_v5.parquet"
)

VALIDATION_V3_PATH = (
    PROCESSED_DATA_DIR
    / "momtsim_v2_validation_features_candidate_v5.parquet"
)

CALIBRATION_V3_PATH = (
    PROCESSED_DATA_DIR
    / "momtsim_v2_calibration_features_candidate_v5.parquet"
)

TEST_V3_PATH = (
    PROCESSED_DATA_DIR
    / "momtsim_v2_test_features_candidate_v5.parquet"
)

print("Train path:", TRAIN_V3_PATH)
print("Validation path:", VALIDATION_V3_PATH)
print("Calibration path:", CALIBRATION_V3_PATH)
print("Test path:", TEST_V3_PATH)


In [ ]:
# ============================================================
# 06.5 SAVE TEMPORAL SPLITS CANDIDATE V5
# ============================================================

train_v3_lf.sink_parquet(
    str(TRAIN_V3_PATH),
    compression="zstd",
)

validation_v3_lf.sink_parquet(
    str(VALIDATION_V3_PATH),
    compression="zstd",
)

calibration_v3_lf.sink_parquet(
    str(CALIBRATION_V3_PATH),
    compression="zstd",
)

test_v3_lf.sink_parquet(
    str(TEST_V3_PATH),
    compression="zstd",
)

print(
    "Train V3 tersimpan:",
    TRAIN_V3_PATH,
)

print(
    "Validation V3 tersimpan:",
    VALIDATION_V3_PATH,
)

print(
    "Calibration V3 tersimpan:",
    CALIBRATION_V3_PATH,
)

print(
    "Test V3 tersimpan:",
    TEST_V3_PATH,
)

In [ ]:
del base_history_df
del initiator_step_df
del initiator_6h_df
del initiator_24h_df
del recipient_step_df
del pair_step_df

gc.collect()

print("Intermediate memory dibersihkan.")

In [ ]:
# ============================================================
# 07.0 BASE FEATURE CONTRACT V3 FOR CANDIDATE V5
# ============================================================

V3_NUMERIC_FEATURES = [
    # --------------------------------------------------------
    # Nominal transaksi
    # --------------------------------------------------------
    "amount",
    "amount_log1p",

    # --------------------------------------------------------
    # Kondisi saldo pengirim
    # Fitur saldo penerima sengaja tidak digunakan
    # karena umumnya tidak tersedia dalam e-statement.
    # --------------------------------------------------------
    "old_bal_initiator",
    "new_bal_initiator",
    "initiator_balance_change",
    "amount_to_initiator_balance_ratio",
    "initiator_balance_drop_ratio",
    "initiator_old_balance_non_positive",
    "initiator_new_balance_negative",
    "initiator_balance_error",
    "remaining_balance_ratio",

    # --------------------------------------------------------
    # Waktu
    # 1 step MoMTSim = 1 jam
    # --------------------------------------------------------
    "hour_of_day",
    "is_night",
    "hour_sin",
    "hour_cos",

    # --------------------------------------------------------
    # Histori pengirim
    # --------------------------------------------------------
    "has_initiator_history",
    "initiator_tx_count_prior",
    "initiator_amount_mean_prior",
    "initiator_amount_std_prior",
    "initiator_amount_max_prior",
    "amount_vs_initiator_mean",
    "amount_zscore_initiator",
    "amount_vs_initiator_max",
    "amount_log_ratio_to_mean",

    # --------------------------------------------------------
    # Aktivitas terbaru
    # --------------------------------------------------------
    "initiator_tx_count_6h",
    "initiator_amount_sum_6h",
    "initiator_tx_count_24h",
    "initiator_amount_sum_24h",
    "tx_velocity_6h_vs_24h",
    "amount_vs_recent_mean_6h",
    "amount_vs_recent_mean_24h",
    "steps_since_initiator_transaction",
    "is_rapid_initiator_repeat",

    # --------------------------------------------------------
    # Histori penerima dan pasangan
    # --------------------------------------------------------
    "recipient_tx_count_prior",
    "recipient_history_log1p",
    "pair_tx_count_prior",
    "pair_history_log1p",
    "is_new_recipient",
    "steps_since_pair_transaction",
    "is_rapid_pair_repeat",

    # --------------------------------------------------------
    # Cold-start
    # --------------------------------------------------------
    "is_cold_start",
]

# Kolom ini tidak masuk ke model.
# Kolom dipertahankan untuk audit hasil prediksi.
V3_AUDIT_COLUMNS = [
    "step",
    "initiator",
    "recipient",
    "transaction_type",
]

V3_TARGET = "target"

print(
    "Jumlah fitur numerik V3:",
    len(V3_NUMERIC_FEATURES),
)

In [ ]:
# ============================================================
# 07.1 LOAD BASE FEATURE SPLIT FOR CANDIDATE V5
# ============================================================

def load_v3_split(path):
    selected_columns = (
        V3_AUDIT_COLUMNS
        + V3_NUMERIC_FEATURES
        + [V3_TARGET]
    )

    lazy_split = pl.scan_parquet(
        str(path)
    )

    available_columns = (
        lazy_split
        .collect_schema()
        .names()
    )

    missing_columns = [
        column
        for column in selected_columns
        if column not in available_columns
    ]

    if missing_columns:
        raise ValueError(
            f"Kolom V3 tidak ditemukan pada {path}: "
            f"{missing_columns}"
        )

    numeric_expressions = [
        pl.col(column)
        .cast(
            pl.Float32,
            strict=False,
        )
        .fill_nan(0)
        .fill_null(0)
        .alias(column)
        for column in V3_NUMERIC_FEATURES
    ]

    loaded_df = (
        lazy_split
        .select(selected_columns)
        .with_columns(
            numeric_expressions
            + [
                pl.col("step")
                .cast(pl.Int64),

                pl.col("initiator")
                .cast(pl.Int64)
                .fill_null(-1),

                pl.col("recipient")
                .cast(pl.String)
                .fill_null("UNKNOWN"),

                pl.col("transaction_type")
                .cast(pl.String)
                .fill_null("UNKNOWN"),

                pl.col(V3_TARGET)
                .cast(pl.Int8)
                .alias(V3_TARGET),
            ]
        )
        .collect(
            engine="streaming"
        )
        .to_pandas()
    )

    return loaded_df

In [ ]:
# ============================================================
# 07.2 LOAD FOUR TEMPORAL SPLITS FOR CANDIDATE V5
# ============================================================

train_v3_df = load_v3_split(
    TRAIN_V3_PATH
)

validation_v3_df = load_v3_split(
    VALIDATION_V3_PATH
)

calibration_v3_df = load_v3_split(
    CALIBRATION_V3_PATH
)

test_v3_df = load_v3_split(
    TEST_V3_PATH
)

print(
    "Train V3:",
    train_v3_df.shape,
)

print(
    "Validation V3:",
    validation_v3_df.shape,
)

print(
    "Calibration V3:",
    calibration_v3_df.shape,
)

print(
    "Test V3:",
    test_v3_df.shape,
)

In [ ]:
# ============================================================
# 07.3A AUDIT LOADED V5 DATASETS
# ============================================================

loaded_split_summary_v3 = pd.DataFrame([
    {
        "split": "train",
        "rows": len(train_v3_df),
        "fraud_count": int(
            train_v3_df[V3_TARGET].sum()
        ),
        "fraud_rate": float(
            train_v3_df[V3_TARGET].mean()
        ),
        "step_start": int(
            train_v3_df["step"].min()
        ),
        "step_end": int(
            train_v3_df["step"].max()
        ),
    },
    {
        "split": "validation",
        "rows": len(validation_v3_df),
        "fraud_count": int(
            validation_v3_df[V3_TARGET].sum()
        ),
        "fraud_rate": float(
            validation_v3_df[V3_TARGET].mean()
        ),
        "step_start": int(
            validation_v3_df["step"].min()
        ),
        "step_end": int(
            validation_v3_df["step"].max()
        ),
    },
    {
        "split": "calibration",
        "rows": len(calibration_v3_df),
        "fraud_count": int(
            calibration_v3_df[V3_TARGET].sum()
        ),
        "fraud_rate": float(
            calibration_v3_df[V3_TARGET].mean()
        ),
        "step_start": int(
            calibration_v3_df["step"].min()
        ),
        "step_end": int(
            calibration_v3_df["step"].max()
        ),
    },
    {
        "split": "test",
        "rows": len(test_v3_df),
        "fraud_count": int(
            test_v3_df[V3_TARGET].sum()
        ),
        "fraud_rate": float(
            test_v3_df[V3_TARGET].mean()
        ),
        "step_start": int(
            test_v3_df["step"].min()
        ),
        "step_end": int(
            test_v3_df["step"].max()
        ),
    },
])

display(
    loaded_split_summary_v3
)

for split_name, split_df in [
    ("train", train_v3_df),
    ("validation", validation_v3_df),
    ("calibration", calibration_v3_df),
    ("test", test_v3_df),
]:
    transaction_types = (
        split_df["transaction_type"]
        .dropna()
        .unique()
        .tolist()
    )

    print(
        split_name,
        "transaction types:",
        transaction_types,
    )

    if transaction_types != ["TRANSFER"]:
        raise ValueError(
            f"Split {split_name} masih memiliki "
            f"tipe selain TRANSFER: "
            f"{transaction_types}"
        )

In [ ]:
# ============================================================
# 07.3B BUILD BASE MATRICES FOR CANDIDATE V5
# ============================================================

def build_model_matrix_v3(
    dataframe,
):
    X = (
        dataframe[
            V3_NUMERIC_FEATURES
        ]
        .copy()
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .fillna(0)
        .astype("float32")
    )

    y = (
        dataframe[V3_TARGET]
        .astype("int8")
        .copy()
    )

    audit = (
        dataframe[
            V3_AUDIT_COLUMNS
        ]
        .copy()
    )

    return X, y, audit


(
    X_train_v3,
    y_train_v3,
    train_audit_v3,
) = build_model_matrix_v3(
    train_v3_df
)

(
    X_validation_v3,
    y_validation_v3,
    validation_audit_v3,
) = build_model_matrix_v3(
    validation_v3_df
)

(
    X_calibration_v3,
    y_calibration_v3,
    calibration_audit_v3,
) = build_model_matrix_v3(
    calibration_v3_df
)

(
    X_test_v3,
    y_test_v3,
    test_audit_v3,
) = build_model_matrix_v3(
    test_v3_df
)

In [ ]:
# ============================================================
# 07.3C VALIDATE BASE MATRICES FOR CANDIDATE V5
# ============================================================

matrix_summary_v3 = pd.DataFrame([
    {
        "split": "train",
        "rows": X_train_v3.shape[0],
        "features": X_train_v3.shape[1],
        "fraud_rate": y_train_v3.mean(),
        "missing_values": int(
            X_train_v3
            .isna()
            .sum()
            .sum()
        ),
        "infinite_values": int(
            np.isinf(
                X_train_v3.to_numpy()
            ).sum()
        ),
    },
    {
        "split": "validation",
        "rows": X_validation_v3.shape[0],
        "features": X_validation_v3.shape[1],
        "fraud_rate": y_validation_v3.mean(),
        "missing_values": int(
            X_validation_v3
            .isna()
            .sum()
            .sum()
        ),
        "infinite_values": int(
            np.isinf(
                X_validation_v3.to_numpy()
            ).sum()
        ),
    },
    {
        "split": "calibration",
        "rows": X_calibration_v3.shape[0],
        "features": X_calibration_v3.shape[1],
        "fraud_rate": y_calibration_v3.mean(),
        "missing_values": int(
            X_calibration_v3
            .isna()
            .sum()
            .sum()
        ),
        "infinite_values": int(
            np.isinf(
                X_calibration_v3.to_numpy()
            ).sum()
        ),
    },
    {
        "split": "test",
        "rows": X_test_v3.shape[0],
        "features": X_test_v3.shape[1],
        "fraud_rate": y_test_v3.mean(),
        "missing_values": int(
            X_test_v3
            .isna()
            .sum()
            .sum()
        ),
        "infinite_values": int(
            np.isinf(
                X_test_v3.to_numpy()
            ).sum()
        ),
    },
])

display(
    matrix_summary_v3
)

assert list(X_train_v3.columns) == V3_NUMERIC_FEATURES
assert list(X_validation_v3.columns) == V3_NUMERIC_FEATURES
assert list(X_calibration_v3.columns) == V3_NUMERIC_FEATURES
assert list(X_test_v3.columns) == V3_NUMERIC_FEATURES

assert (
    matrix_summary_v3["missing_values"]
    == 0
).all()

assert (
    matrix_summary_v3["infinite_values"]
    == 0
).all()

print(
    "Matriks Candidate V3 siap untuk training."
)

In [ ]:
# ============================================================
# 07.3D CANDIDATE V5 CLASS-PRIOR BASELINE
# ============================================================

class_prior_baseline_rows_v5 = []

for split_name, target in [
    ("train", y_train_v3),
    ("validation", y_validation_v3),
    ("calibration", y_calibration_v3),
    ("test", y_test_v3),
]:
    fraud_rate = float(target.mean())

    class_prior_baseline_rows_v5.append({
        "split": split_name,
        "rows": int(len(target)),
        "fraud_rate": fraud_rate,
        "random_ranker_pr_auc": fraud_rate,
        "all_positive_precision": fraud_rate,
        "all_positive_recall": 1.0,
        "all_positive_alert_rate": 1.0,
        "all_positive_false_positive_rate": 1.0,
    })

class_prior_baseline_v5 = pd.DataFrame(
    class_prior_baseline_rows_v5
)

display(class_prior_baseline_v5)

print(
    "PR-AUC Candidate V5 harus dibandingkan "
    "dengan fraud rate pada split yang sama."
)


In [ ]:
# ============================================================
# 07.3 CANDIDATE V5 THRESHOLD AND EVALUATION UTILITIES
# ============================================================

def select_threshold_v5(
    model_name,
    y_true,
    probabilities,
):
    """
    Memilih threshold hanya dari calibration set.

    Prioritas:
    1. Memenuhi seluruh target.
    2. Jika gagal, pertahankan recall minimum.
    3. Pada recall minimum, pilih FPR terendah.
    """
    operating_points = build_operating_point_table(
        y_true=y_true,
        probabilities=probabilities,
    )

    pr_auc = float(
        average_precision_score(
            y_true,
            probabilities,
        )
    )

    feasible_points = operating_points[
        (
            operating_points["precision"]
            >= TARGET_PRECISION
        )
        &
        (
            operating_points["recall"]
            >= TARGET_RECALL
        )
        &
        (
            operating_points["f1"]
            >= TARGET_F1
        )
        &
        (
            operating_points["false_positive_rate"]
            <= TARGET_FPR
        )
    ].copy()

    if not feasible_points.empty:
        selected = (
            feasible_points
            .sort_values(
                [
                    "f1",
                    "false_positive_rate",
                    "precision",
                    "threshold",
                ],
                ascending=[
                    False,
                    True,
                    False,
                    False,
                ],
            )
            .iloc[0]
        )

        selection_strategy = "ALL_TARGETS_ACHIEVED"

    else:
        recall_guardrail = operating_points[
            operating_points["recall"]
            >= TARGET_RECALL
        ].copy()

        if not recall_guardrail.empty:
            selected = (
                recall_guardrail
                .sort_values(
                    [
                        "false_positive_rate",
                        "precision",
                        "f1",
                        "threshold",
                    ],
                    ascending=[
                        True,
                        False,
                        False,
                        False,
                    ],
                )
                .iloc[0]
            )

            selection_strategy = (
                "MINIMUM_FPR_AT_RECALL_GUARDRAIL"
            )

        else:
            selected = (
                operating_points
                .sort_values(
                    [
                        "f1",
                        "precision",
                        "recall",
                    ],
                    ascending=[
                        False,
                        False,
                        False,
                    ],
                )
                .iloc[0]
            )

            selection_strategy = "BEST_AVAILABLE_F1"

    target_achieved = (
        pr_auc >= TARGET_PR_AUC
        and selected["precision"] >= TARGET_PRECISION
        and selected["recall"] >= TARGET_RECALL
        and selected["f1"] >= TARGET_F1
        and selected["false_positive_rate"] <= TARGET_FPR
    )

    summary = pd.DataFrame([
        {
            "model": model_name,
            "status": (
                "TARGET ACHIEVED"
                if target_achieved
                else "TARGET NOT ACHIEVED"
            ),
            "selection_strategy": selection_strategy,
            "pr_auc": pr_auc,
            "threshold": float(
                selected["threshold"]
            ),
            "precision": float(
                selected["precision"]
            ),
            "recall": float(
                selected["recall"]
            ),
            "f1": float(
                selected["f1"]
            ),
            "false_positive_rate": float(
                selected["false_positive_rate"]
            ),
            "true_positive": int(
                selected["true_positive"]
            ),
            "false_positive": int(
                selected["false_positive"]
            ),
            "false_negative": int(
                selected["false_negative"]
            ),
            "true_negative": int(
                selected["true_negative"]
            ),
        }
    ])

    return summary, operating_points


def evaluate_fixed_threshold_v5(
    model_name,
    y_true,
    probabilities,
    threshold,
):
    """
    Mengevaluasi probabilitas dengan threshold terkunci.
    """
    y_true = np.asarray(
        y_true,
        dtype=np.int8,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    predictions = (
        probabilities >= threshold
    ).astype(np.int8)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    precision = (
        tp / max(tp + fp, 1)
    )

    recall = (
        tp / max(tp + fn, 1)
    )

    f1 = (
        2 * precision * recall
        / max(
            precision + recall,
            1e-12,
        )
    )

    false_positive_rate = (
        fp / max(fp + tn, 1)
    )

    fraud_rate = float(
        y_true.mean()
    )

    pr_auc = float(
        average_precision_score(
            y_true,
            probabilities,
        )
    )

    target_achieved = (
        pr_auc >= TARGET_PR_AUC
        and precision >= TARGET_PRECISION
        and recall >= TARGET_RECALL
        and f1 >= TARGET_F1
        and false_positive_rate <= TARGET_FPR
    )

    return pd.DataFrame([
        {
            "model": model_name,
            "status": (
                "TARGET ACHIEVED"
                if target_achieved
                else "TARGET NOT ACHIEVED"
            ),
            "rows": len(y_true),
            "fraud_rate": fraud_rate,
            "pr_auc": pr_auc,
            "threshold": float(threshold),
            "precision": float(precision),
            "precision_lift": float(
                precision
                / max(fraud_rate, 1e-12)
            ),
            "recall": float(recall),
            "f1": float(f1),
            "false_positive_rate": float(
                false_positive_rate
            ),
            "alert_rate": float(
                predictions.mean()
            ),
            "true_positive": int(tp),
            "false_positive": int(fp),
            "false_negative": int(fn),
            "true_negative": int(tn),
        }
    ])

In [ ]:
# ============================================================
# 08.0 CANDIDATE V5 FEATURE DIAGNOSTIC
# ============================================================

validation_base_rate_v5 = float(
    y_validation_v3.mean()
)

feature_diagnostic_rows_v5 = []

for feature in V3_NUMERIC_FEATURES:
    train_series = X_train_v3[feature]
    validation_series = X_validation_v3[feature]

    train_unique = int(
        train_series.nunique(
            dropna=False
        )
    )

    validation_unique = int(
        validation_series.nunique(
            dropna=False
        )
    )

    train_std = float(
        train_series.std(
            ddof=0
        )
    )

    validation_std = float(
        validation_series.std(
            ddof=0
        )
    )

    train_zero_rate = float(
        np.isclose(
            train_series.to_numpy(),
            0.0,
        ).mean()
    )

    validation_zero_rate = float(
        np.isclose(
            validation_series.to_numpy(),
            0.0,
        ).mean()
    )

    is_constant_train = (
        train_unique <= 1
    )

    is_constant_validation = (
        validation_unique <= 1
    )

    if (
        is_constant_train
        or is_constant_validation
    ):
        best_univariate_ap = (
            validation_base_rate_v5
        )

        positive_ap = np.nan
        negative_ap = np.nan

        if (
            is_constant_train
            and is_constant_validation
        ):
            risk_direction = (
                "constant_train_and_validation"
            )

        elif is_constant_train:
            risk_direction = (
                "constant_train"
            )

        else:
            risk_direction = (
                "constant_validation"
            )

    else:
        positive_ap = float(
            average_precision_score(
                y_validation_v3,
                validation_series,
            )
        )

        negative_ap = float(
            average_precision_score(
                y_validation_v3,
                -validation_series,
            )
        )

        if positive_ap >= negative_ap:
            best_univariate_ap = (
                positive_ap
            )

            risk_direction = (
                "higher_is_risk"
            )

        else:
            best_univariate_ap = (
                negative_ap
            )

            risk_direction = (
                "lower_is_risk"
            )

    validation_ap_lift = float(
        best_univariate_ap
        / max(
            validation_base_rate_v5,
            1e-12,
        )
    )

    feature_diagnostic_rows_v5.append({
        "feature": feature,
        "train_unique": (
            train_unique
        ),
        "validation_unique": (
            validation_unique
        ),
        "train_std": (
            train_std
        ),
        "validation_std": (
            validation_std
        ),
        "train_zero_rate": (
            train_zero_rate
        ),
        "validation_zero_rate": (
            validation_zero_rate
        ),
        "positive_univariate_ap": (
            positive_ap
        ),
        "negative_univariate_ap": (
            negative_ap
        ),
        "validation_univariate_ap": (
            best_univariate_ap
        ),
        "validation_ap_lift": (
            validation_ap_lift
        ),
        "is_constant_train": (
            is_constant_train
        ),
        "is_constant_validation": (
            is_constant_validation
        ),
        "risk_direction": (
            risk_direction
        ),
    })

feature_diagnostic_v5 = (
    pd.DataFrame(
        feature_diagnostic_rows_v5
    )
    .sort_values(
        [
            "validation_univariate_ap",
            "train_unique",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

constant_features_v5 = (
    feature_diagnostic_v5.loc[
        feature_diagnostic_v5[
            "train_unique"
        ] <= 1,
        "feature",
    ]
    .tolist()
)

non_generalizing_features_v5 = (
    feature_diagnostic_v5.loc[
        (
            feature_diagnostic_v5[
                "train_unique"
            ] > 1
        )
        &
        (
            feature_diagnostic_v5[
                "validation_unique"
            ] <= 1
        ),
        "feature",
    ]
    .tolist()
)

constant_validation_features_v5 = (
    feature_diagnostic_v5.loc[
        feature_diagnostic_v5[
            "validation_unique"
        ] <= 1,
        "feature",
    ]
    .tolist()
)

weak_signal_features_v5 = (
    feature_diagnostic_v5.loc[
        feature_diagnostic_v5[
            "validation_ap_lift"
        ] <= 1.01,
        "feature",
    ]
    .tolist()
)

strongest_features_v5 = (
    feature_diagnostic_v5.loc[
        feature_diagnostic_v5[
            "validation_ap_lift"
        ] > 1.05,
        [
            "feature",
            "validation_univariate_ap",
            "validation_ap_lift",
            "risk_direction",
        ],
    ]
    .copy()
)

display(
    feature_diagnostic_v5
)

print(
    "Validation fraud rate:",
    validation_base_rate_v5,
)

print(
    "Jumlah fitur konstan pada train:",
    len(
        constant_features_v5
    ),
)

print(
    "Fitur konstan pada train:",
    constant_features_v5,
)

print(
    "Jumlah fitur konstan pada validation:",
    len(
        constant_validation_features_v5
    ),
)

print(
    "Fitur konstan pada validation:",
    constant_validation_features_v5,
)

print(
    "Jumlah fitur berubah pada train "
    "tetapi konstan pada validation:",
    len(
        non_generalizing_features_v5
    ),
)

print(
    "Fitur tidak bergeneralisasi:",
    non_generalizing_features_v5,
)

print(
    "Jumlah fitur dengan AP lift <= 1.01:",
    len(
        weak_signal_features_v5
    ),
)

print(
    "Fitur dengan sinyal univariat lemah:",
    weak_signal_features_v5,
)

print(
    "Fitur dengan AP lift > 1.05:"
)

display(
    strongest_features_v5
)

In [ ]:
# ============================================================
# 08.1 CANDIDATE V5 DEPLOYABLE INTERACTION FEATURES
# ============================================================

V5_DERIVED_FEATURES = [
    "initiator_history_log1p",
    "amount_log_ratio_to_prior_max",
    "amount_excess_over_prior_max_ratio",
    "is_amount_above_prior_max",
    "pair_share_of_initiator_history",
    "current_amount_share_6h",
    "current_amount_share_24h",
    "recent_sum_6h_share_24h",
    "steps_since_initiator_log1p",
    "steps_since_pair_log1p",
    "new_recipient_amount_log",
    "cold_start_amount_log",
    "initiator_balance_error_log1p",
    "is_balance_nearly_depleted",
    "amount_to_balance_ratio_capped",
]


def add_v5_derived_features(
    dataframe,
):
    output = dataframe.copy()

    amount = (
        output["amount"]
        .clip(lower=0)
    )

    amount_log = np.log1p(
        amount
    )

    prior_max = (
        output[
            "initiator_amount_max_prior"
        ]
        .clip(lower=0)
    )

    has_history = (
        output[
            "has_initiator_history"
        ] > 0
    ).astype("float32")

    history_count = (
        output[
            "initiator_tx_count_prior"
        ]
        .clip(lower=0)
    )

    output[
        "initiator_history_log1p"
    ] = np.log1p(
        history_count
    )

    output[
        "amount_log_ratio_to_prior_max"
    ] = (
        amount_log
        - np.log1p(prior_max)
    ) * has_history

    output[
        "amount_excess_over_prior_max_ratio"
    ] = (
        (
            amount - prior_max
        )
        .clip(lower=0)
        /
        (
            prior_max + 1.0
        )
    ) * has_history

    output[
        "is_amount_above_prior_max"
    ] = (
        (
            output[
                "has_initiator_history"
            ] > 0
        )
        &
        (
            amount > prior_max
        )
    ).astype("float32")

    output[
        "pair_share_of_initiator_history"
    ] = (
        output[
            "pair_tx_count_prior"
        ]
        /
        (
            history_count + 1.0
        )
    )

    output[
        "current_amount_share_6h"
    ] = (
        amount
        /
        (
            output[
                "initiator_amount_sum_6h"
            ]
            + amount
            + 1.0
        )
    )

    output[
        "current_amount_share_24h"
    ] = (
        amount
        /
        (
            output[
                "initiator_amount_sum_24h"
            ]
            + amount
            + 1.0
        )
    )

    output[
        "recent_sum_6h_share_24h"
    ] = (
        output[
            "initiator_amount_sum_6h"
        ]
        /
        (
            output[
                "initiator_amount_sum_24h"
            ]
            + 1.0
        )
    )

    output[
        "steps_since_initiator_log1p"
    ] = np.where(
        output[
            "steps_since_initiator_transaction"
        ] >= 0,
        np.log1p(
            output[
                "steps_since_initiator_transaction"
            ].clip(lower=0)
        ),
        0.0,
    )

    output[
        "steps_since_pair_log1p"
    ] = np.where(
        output[
            "steps_since_pair_transaction"
        ] >= 0,
        np.log1p(
            output[
                "steps_since_pair_transaction"
            ].clip(lower=0)
        ),
        0.0,
    )

    output[
        "new_recipient_amount_log"
    ] = (
        output["is_new_recipient"]
        * amount_log
    )

    output[
        "cold_start_amount_log"
    ] = (
        output["is_cold_start"]
        * amount_log
    )

    output[
        "initiator_balance_error_log1p"
    ] = np.log1p(
        output[
            "initiator_balance_error"
        ].clip(lower=0)
    )

    old_balance = (
        output["old_bal_initiator"]
        .abs()
    )

    remaining_ratio = (
        output["new_bal_initiator"]
        /
        (
            old_balance + 1.0
        )
    )

    output[
        "is_balance_nearly_depleted"
    ] = (
        (
            output[
                "old_bal_initiator"
            ] > 0
        )
        &
        (
            remaining_ratio <= 0.05
        )
    ).astype("float32")

    output[
        "amount_to_balance_ratio_capped"
    ] = (
        output[
            "amount_to_initiator_balance_ratio"
        ]
        .clip(
            lower=0,
            upper=20,
        )
    )

    output = (
        output
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .fillna(0)
    )

    output[
        V5_DERIVED_FEATURES
    ] = (
        output[
            V5_DERIVED_FEATURES
        ]
        .astype("float32")
    )

    return output


V5_BASE_FEATURES = [
    feature
    for feature in V3_NUMERIC_FEATURES
    if feature not in constant_features_v5
]

V5_NUMERIC_FEATURES = (
    V5_BASE_FEATURES
    + V5_DERIVED_FEATURES
)

X_train_v5 = (
    add_v5_derived_features(
        X_train_v3
    )[V5_NUMERIC_FEATURES]
    .astype("float32")
)

X_validation_v5 = (
    add_v5_derived_features(
        X_validation_v3
    )[V5_NUMERIC_FEATURES]
    .astype("float32")
)

X_calibration_v5 = (
    add_v5_derived_features(
        X_calibration_v3
    )[V5_NUMERIC_FEATURES]
    .astype("float32")
)

X_test_v5 = (
    add_v5_derived_features(
        X_test_v3
    )[V5_NUMERIC_FEATURES]
    .astype("float32")
)

y_train_v5 = y_train_v3.copy()
y_validation_v5 = y_validation_v3.copy()
y_calibration_v5 = y_calibration_v3.copy()
y_test_v5 = y_test_v3.copy()

print(
    "Fitur dasar V3:",
    len(V3_NUMERIC_FEATURES),
)

print(
    "Fitur konstan dibuang:",
    len(constant_features_v5),
)

print(
    "Fitur tambahan V5:",
    len(V5_DERIVED_FEATURES),
)

print(
    "Total fitur V5:",
    len(V5_NUMERIC_FEATURES),
)

In [ ]:
# ============================================================
# 08.2 CANDIDATE V5 MATRIX VALIDATION
# ============================================================

matrix_summary_v5 = []

for (
    split_name,
    matrix,
    target,
) in [
    (
        "train",
        X_train_v5,
        y_train_v5,
    ),
    (
        "validation",
        X_validation_v5,
        y_validation_v5,
    ),
    (
        "calibration",
        X_calibration_v5,
        y_calibration_v5,
    ),
    (
        "test",
        X_test_v5,
        y_test_v5,
    ),
]:
    missing_values = int(
        matrix
        .isna()
        .sum()
        .sum()
    )

    infinite_values = int(
        np.isinf(
            matrix.to_numpy()
        ).sum()
    )

    matrix_summary_v5.append({
        "split": split_name,
        "rows": matrix.shape[0],
        "features": matrix.shape[1],
        "fraud_rate": float(
            target.mean()
        ),
        "missing_values": (
            missing_values
        ),
        "infinite_values": (
            infinite_values
        ),
    })

matrix_summary_v5 = pd.DataFrame(
    matrix_summary_v5
)

display(
    matrix_summary_v5
)

assert (
    list(X_train_v5.columns)
    == V5_NUMERIC_FEATURES
)

assert (
    list(X_validation_v5.columns)
    == V5_NUMERIC_FEATURES
)

assert (
    list(X_calibration_v5.columns)
    == V5_NUMERIC_FEATURES
)

assert (
    list(X_test_v5.columns)
    == V5_NUMERIC_FEATURES
)

assert (
    matrix_summary_v5[
        "missing_values"
    ] == 0
).all()

assert (
    matrix_summary_v5[
        "infinite_values"
    ] == 0
).all()

print(
    "Matriks Candidate V5 siap untuk benchmark."
)

In [ ]:
# ============================================================
# 08.2A CANDIDATE V5 HEAVY-TAIL FEATURE AUDIT
# ============================================================

V5_HEAVY_TAIL_FEATURES = [
    "amount",
    "amount_log1p",
    "old_bal_initiator",
    "new_bal_initiator",
    "initiator_balance_change",
    "amount_to_initiator_balance_ratio",
    "initiator_balance_error",
    "amount_vs_initiator_mean",
    "amount_zscore_initiator",
    "amount_vs_initiator_max",
    "amount_excess_over_prior_max_ratio",
]

heavy_tail_rows_v5 = []

for feature in V5_HEAVY_TAIL_FEATURES:
    if feature not in X_train_v5.columns:
        continue

    values = (
        X_train_v5[feature]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if values.empty:
        continue

    heavy_tail_rows_v5.append({
        "feature": feature,
        "minimum": float(values.min()),
        "p50": float(values.quantile(0.50)),
        "p90": float(values.quantile(0.90)),
        "p99": float(values.quantile(0.99)),
        "p999": float(values.quantile(0.999)),
        "maximum": float(values.max()),
    })

heavy_tail_audit_v5 = pd.DataFrame(
    heavy_tail_rows_v5
)

display(heavy_tail_audit_v5)


In [ ]:
# ============================================================
# 08.3 CANDIDATE V5 CONTROLLED LIGHTGBM BENCHMARK
# ============================================================

V5_COMMON_PARAMS = {
    "objective": "binary",
    "n_estimators": 2200,
    "learning_rate": 0.03,
    "max_bin": 255,
    "class_weight": None,
    "random_state": SEED,
    "n_jobs": -1,
    "force_col_wise": True,
    "verbosity": -1,
    "subsample_freq": 1,
}

V5_CANDIDATE_CONFIGS = {
    "conservative": {
        "num_leaves": 15,
        "max_depth": 5,
        "min_child_samples": 100,
        "min_split_gain": 0.0,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.50,
        "reg_lambda": 3.00,
    },
    "balanced": {
        "num_leaves": 31,
        "max_depth": 8,
        "min_child_samples": 50,
        "min_split_gain": 0.0,
        "subsample": 0.90,
        "colsample_bytree": 1.00,
        "reg_alpha": 0.10,
        "reg_lambda": 1.00,
    },
    "flexible": {
        "num_leaves": 63,
        "max_depth": 10,
        "min_child_samples": 30,
        "min_split_gain": 0.0,
        "subsample": 0.90,
        "colsample_bytree": 1.00,
        "reg_alpha": 0.00,
        "reg_lambda": 0.50,
    },
}

benchmark_models_v5 = {}
benchmark_rows_v5 = []

for (
    config_name,
    config_parameters,
) in V5_CANDIDATE_CONFIGS.items():

    print(
        "\nTraining configuration:",
        config_name,
    )

    model_parameters = {
        **V5_COMMON_PARAMS,
        **config_parameters,
    }

    candidate_model = LGBMClassifier(
        **model_parameters
    )

    candidate_model.fit(
        X_train_v5,
        y_train_v5,
        eval_set=[
            (
                X_validation_v5,
                y_validation_v5,
            )
        ],
        eval_metric="average_precision",
        callbacks=[
            early_stopping(
                stopping_rounds=100,
                first_metric_only=True,
                verbose=True,
            ),
            log_evaluation(
                period=50,
            ),
        ],
    )

    best_iteration = int(
        candidate_model.best_iteration_
        or V5_COMMON_PARAMS[
            "n_estimators"
        ]
    )

    validation_probabilities = (
        candidate_model.predict_proba(
            X_validation_v5,
            num_iteration=best_iteration,
        )[:, 1]
    )

    validation_pr_auc = float(
        average_precision_score(
            y_validation_v5,
            validation_probabilities,
        )
    )

    validation_operating_points = (
        build_operating_point_table(
            y_true=y_validation_v5,
            probabilities=(
                validation_probabilities
            ),
        )
    )

    recall_guardrail = (
        validation_operating_points[
            validation_operating_points[
                "recall"
            ] >= TARGET_RECALL
        ]
        .copy()
    )

    if recall_guardrail.empty:
        selected_validation_point = (
            validation_operating_points
            .sort_values(
                [
                    "f1",
                    "precision",
                ],
                ascending=[
                    False,
                    False,
                ],
            )
            .iloc[0]
        )
    else:
        selected_validation_point = (
            recall_guardrail
            .sort_values(
                [
                    "false_positive_rate",
                    "precision",
                    "f1",
                    "threshold",
                ],
                ascending=[
                    True,
                    False,
                    False,
                    False,
                ],
            )
            .iloc[0]
        )

    split_importance = (
        candidate_model
        .booster_
        .feature_importance(
            importance_type="split"
        )
    )

    used_feature_count = int(
        (
            split_importance > 0
        ).sum()
    )

    benchmark_models_v5[
        config_name
    ] = candidate_model

    benchmark_rows_v5.append({
        "config": config_name,
        "best_iteration": (
            best_iteration
        ),
        "validation_pr_auc": (
            validation_pr_auc
        ),
        "validation_pr_auc_lift": (
            validation_pr_auc
            / max(
                float(
                    y_validation_v5.mean()
                ),
                1e-12,
            )
        ),
        "precision_at_recall_70": float(
            selected_validation_point[
                "precision"
            ]
        ),
        "recall": float(
            selected_validation_point[
                "recall"
            ]
        ),
        "f1": float(
            selected_validation_point[
                "f1"
            ]
        ),
        "fpr_at_recall_70": float(
            selected_validation_point[
                "false_positive_rate"
            ]
        ),
        "threshold_at_recall_70": float(
            selected_validation_point[
                "threshold"
            ]
        ),
        "used_feature_count": (
            used_feature_count
        ),
    })

benchmark_results_v5 = (
    pd.DataFrame(
        benchmark_rows_v5
    )
    .sort_values(
        [
            "validation_pr_auc",
            "fpr_at_recall_70",
            "precision_at_recall_70",
        ],
        ascending=[
            False,
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

best_config_name_v5 = str(
    benchmark_results_v5
    .iloc[0]["config"]
)

lightgbm_v5_selection_model = (
    benchmark_models_v5[
        best_config_name_v5
    ]
)

display(
    benchmark_results_v5
)

print(
    "Selected configuration:",
    best_config_name_v5,
)

In [ ]:
# ============================================================
# 08.4 CANDIDATE V5 REFIT TRAIN AND VALIDATION
# ============================================================

selected_best_iteration_v5 = int(
    lightgbm_v5_selection_model
    .best_iteration_
    or V5_COMMON_PARAMS[
        "n_estimators"
    ]
)

selected_parameters_v5 = {
    **V5_COMMON_PARAMS,
    **V5_CANDIDATE_CONFIGS[
        best_config_name_v5
    ],
}

selected_parameters_v5[
    "n_estimators"
] = selected_best_iteration_v5

X_refit_v5 = pd.concat(
    [
        X_train_v5,
        X_validation_v5,
    ],
    axis=0,
    ignore_index=True,
)

y_refit_v5 = pd.concat(
    [
        y_train_v5,
        y_validation_v5,
    ],
    axis=0,
    ignore_index=True,
)

lightgbm_v5_model = LGBMClassifier(
    **selected_parameters_v5
)

lightgbm_v5_model.fit(
    X_refit_v5,
    y_refit_v5,
)

selection_validation_probabilities_v5 = (
    lightgbm_v5_selection_model
    .predict_proba(
        X_validation_v5,
        num_iteration=(
            selected_best_iteration_v5
        ),
    )[:, 1]
)

selection_validation_pr_auc_v5 = float(
    average_precision_score(
        y_validation_v5,
        selection_validation_probabilities_v5,
    )
)

print(
    "Selected configuration:",
    best_config_name_v5,
)

print(
    "Selected best iteration:",
    selected_best_iteration_v5,
)

print(
    "Refit rows:",
    len(X_refit_v5),
)

print(
    "Selection validation PR-AUC:",
    selection_validation_pr_auc_v5,
)

In [ ]:
# ============================================================
# 09.0 CANDIDATE V5 CALIBRATION THRESHOLD SELECTION
# ============================================================

calibration_probabilities_v5 = (
    lightgbm_v5_model
    .predict_proba(
        X_calibration_v5
    )[:, 1]
)

(
    calibration_summary_v5,
    calibration_operating_points_v5,
) = select_threshold_v5(
    model_name=(
        "LightGBM Candidate V5 Calibration"
    ),
    y_true=y_calibration_v5,
    probabilities=(
        calibration_probabilities_v5
    ),
)

selected_threshold_v5 = float(
    calibration_summary_v5
    .iloc[0]["threshold"]
)

display(
    calibration_summary_v5
)

print(
    "Selected threshold V5:",
    selected_threshold_v5,
)

In [ ]:
# ============================================================
# 10.0 CANDIDATE V5 FINAL TEST EVALUATION
# ============================================================

test_probabilities_v5 = (
    lightgbm_v5_model
    .predict_proba(
        X_test_v5
    )[:, 1]
)

test_summary_v5 = (
    evaluate_fixed_threshold_v5(
        model_name=(
            "LightGBM Candidate V5 Final Test"
        ),
        y_true=y_test_v5,
        probabilities=(
            test_probabilities_v5
        ),
        threshold=(
            selected_threshold_v5
        ),
    )
)

test_predictions_v5 = (
    test_probabilities_v5
    >= selected_threshold_v5
).astype(np.int8)

tn_v5, fp_v5, fn_v5, tp_v5 = (
    confusion_matrix(
        y_test_v5,
        test_predictions_v5,
        labels=[0, 1],
    )
    .ravel()
)

test_confusion_matrix_v5 = pd.DataFrame(
    [
        {
            "actual_class": "NORMAL",
            "predicted_normal": int(
                tn_v5
            ),
            "predicted_attention": int(
                fp_v5
            ),
        },
        {
            "actual_class": (
                "SYNTHETIC_FRAUD"
            ),
            "predicted_normal": int(
                fn_v5
            ),
            "predicted_attention": int(
                tp_v5
            ),
        },
    ]
)

display(
    test_summary_v5
)

display(
    test_confusion_matrix_v5
)

print(
    classification_report(
        y_test_v5,
        test_predictions_v5,
        labels=[0, 1],
        target_names=[
            "normal",
            "synthetic_fraud",
        ],
        digits=6,
        zero_division=0,
    )
)

# CANDIDATE V4/V1 VS CANDIDATE V5/V2 COMPARISON
# ============================================================

v5_test_row = test_summary_v5.iloc[0]

v5_current_metrics = {
    "fraud_rate": float(v5_test_row["fraud_rate"]),
    "pr_auc": float(v5_test_row["pr_auc"]),
    "pr_auc_lift": float(
        v5_test_row["pr_auc"]
        / max(v5_test_row["fraud_rate"], 1e-12)
    ),
    "precision": float(v5_test_row["precision"]),
    "precision_lift": float(
        v5_test_row["precision_lift"]
    ),
    "recall": float(v5_test_row["recall"]),
    "f1": float(v5_test_row["f1"]),
    "false_positive_rate": float(
        v5_test_row["false_positive_rate"]
    ),
    "alert_rate": float(v5_test_row["alert_rate"]),
}

comparison_rows_v5 = []

if V4_V1_BASELINE_TEST_METRICS is not None:
    v4_comparison_row = {
        "model": "Candidate V4 / MoMTSim V1",
        **V4_V1_BASELINE_TEST_METRICS,
    }

    if (
        v4_comparison_row.get("pr_auc") is not None
        and v4_comparison_row.get("fraud_rate") is not None
    ):
        v4_comparison_row["pr_auc_lift"] = (
            v4_comparison_row["pr_auc"]
            / max(
                v4_comparison_row["fraud_rate"],
                1e-12,
            )
        )

    comparison_rows_v5.append(v4_comparison_row)
else:
    print(
        "WARNING: baseline V4/V1 tidak tersedia. "
        "Comparison hanya menampilkan V5/V2."
    )

comparison_rows_v5.append({
    "model": "Candidate V5 / MoMTSim V2",
    **v5_current_metrics,
})

comparison_v4_v5 = pd.DataFrame(
    comparison_rows_v5
)

display(comparison_v4_v5)

print(
    "Raw PR-AUC V1 dan V2 tidak dibandingkan "
    "tanpa fraud rate dan PR-AUC lift."
)


In [ ]:
# ============================================================
# 10.0A CANDIDATE V5 COLD-START SEGMENT EVALUATION
# ============================================================

if len(test_v3_df) != len(y_test_v5):
    raise ValueError(
        "Jumlah baris source test dan target test berbeda."
    )

test_segment_source_v5 = pd.DataFrame({
    "target": np.asarray(
        y_test_v5,
        dtype=np.int8,
    ),
    "probability": np.asarray(
        test_probabilities_v5,
        dtype=np.float64,
    ),
    "is_cold_start": (
        test_v3_df["is_cold_start"]
        .astype("int8")
        .to_numpy()
    ),
    "initiator_tx_count_prior": (
        test_v3_df["initiator_tx_count_prior"]
        .astype("float32")
        .to_numpy()
    ),
})

cold_start_segment_rows_v5 = []

for segment_name, segment_value in [
    ("cold_start", 1),
    ("history_available", 0),
]:
    segment_df = test_segment_source_v5.loc[
        test_segment_source_v5["is_cold_start"]
        == segment_value
    ]

    if segment_df.empty:
        continue

    segment_metrics = (
        evaluate_fixed_threshold_v5(
            model_name=segment_name,
            y_true=segment_df["target"],
            probabilities=segment_df["probability"],
            threshold=selected_threshold_v5,
        )
        .iloc[0]
        .to_dict()
    )

    segment_metrics["segment"] = segment_name
    segment_metrics["segment_share"] = float(
        len(segment_df)
        / len(test_segment_source_v5)
    )
    segment_metrics[
        "median_prior_transactions"
    ] = float(
        segment_df[
            "initiator_tx_count_prior"
        ].median()
    )
    segment_metrics["pr_auc_lift"] = float(
        segment_metrics["pr_auc"]
        / max(
            segment_metrics["fraud_rate"],
            1e-12,
        )
    )

    cold_start_segment_rows_v5.append(
        segment_metrics
    )

cold_start_segment_metrics_v5 = pd.DataFrame(
    cold_start_segment_rows_v5
)

display(cold_start_segment_metrics_v5)


In [ ]:
# ============================================================
# 10.1 CANDIDATE V5 PRECISION AND RECALL AT TOP K
# ============================================================

def calculate_top_k_metrics(
    y_true,
    probabilities,
    fractions,
):
    y_true = np.asarray(
        y_true,
        dtype=np.int8,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    sorted_indices = np.argsort(
        -probabilities,
        kind="mergesort",
    )

    total_positive = max(
        int(y_true.sum()),
        1,
    )

    rows = []

    for fraction in fractions:
        k = max(
            int(
                np.ceil(
                    len(y_true)
                    * fraction
                )
            ),
            1,
        )

        selected_indices = (
            sorted_indices[:k]
        )

        selected_positive = int(
            y_true[
                selected_indices
            ].sum()
        )

        rows.append({
            "top_fraction": fraction,
            "top_percent": (
                fraction * 100
            ),
            "transactions": k,
            "true_positive": (
                selected_positive
            ),
            "precision_at_k": (
                selected_positive
                / k
            ),
            "recall_at_k": (
                selected_positive
                / total_positive
            ),
        })

    return pd.DataFrame(rows)


top_k_metrics_v5 = (
    calculate_top_k_metrics(
        y_true=y_test_v5,
        probabilities=(
            test_probabilities_v5
        ),
        fractions=[
            0.01,
            0.05,
            0.10,
            0.20,
        ],
    )
)

display(
    top_k_metrics_v5
)

In [ ]:
# ============================================================
# 10.2 CANDIDATE V5 CALIBRATION-LOCKED TOP-FRACTION
# OPERATING POINTS
# ============================================================

GLOBAL_TOP_FRACTIONS_V5 = [
    0.01,
    0.02,
    0.05,
    0.10,
]


def calculate_fixed_threshold_metrics_v5(
    y_true,
    probabilities,
    threshold,
):
    y_true = np.asarray(
        y_true,
        dtype=np.int8,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    predictions = (
        probabilities >= threshold
    ).astype(np.int8)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    precision = (
        tp / max(tp + fp, 1)
    )

    recall = (
        tp / max(tp + fn, 1)
    )

    f1 = (
        2 * precision * recall
        / max(
            precision + recall,
            1e-12,
        )
    )

    false_positive_rate = (
        fp / max(fp + tn, 1)
    )

    return {
        "rows": len(y_true),
        "fraud_rate": float(
            y_true.mean()
        ),
        "alert_rate": float(
            predictions.mean()
        ),
        "precision": float(
            precision
        ),
        "recall": float(
            recall
        ),
        "f1": float(
            f1
        ),
        "false_positive_rate": float(
            false_positive_rate
        ),
        "true_positive": int(tp),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_negative": int(tn),
    }


top_fraction_rows_v5 = []

for configured_fraction in (
    GLOBAL_TOP_FRACTIONS_V5
):
    threshold = float(
        np.quantile(
            calibration_probabilities_v5,
            1.0 - configured_fraction,
        )
    )

    calibration_metrics = (
        calculate_fixed_threshold_metrics_v5(
            y_true=y_calibration_v5,
            probabilities=(
                calibration_probabilities_v5
            ),
            threshold=threshold,
        )
    )

    test_metrics = (
        calculate_fixed_threshold_metrics_v5(
            y_true=y_test_v5,
            probabilities=(
                test_probabilities_v5
            ),
            threshold=threshold,
        )
    )

    top_fraction_rows_v5.append({
        "configured_top_fraction": (
            configured_fraction
        ),
        "configured_top_percent": (
            configured_fraction * 100
        ),
        "threshold": threshold,
        "split": "calibration",
        **calibration_metrics,
    })

    top_fraction_rows_v5.append({
        "configured_top_fraction": (
            configured_fraction
        ),
        "configured_top_percent": (
            configured_fraction * 100
        ),
        "threshold": threshold,
        "split": "test",
        **test_metrics,
    })

top_fraction_operating_points_v5 = (
    pd.DataFrame(
        top_fraction_rows_v5
    )
    .sort_values(
        [
            "configured_top_fraction",
            "split",
        ]
    )
    .reset_index(drop=True)
)

display(
    top_fraction_operating_points_v5
)

In [ ]:
# ============================================================
# 10.3 CANDIDATE V5 GLOBAL SCORE PERCENTILE
# ============================================================

calibration_score_reference_v5 = np.sort(
    np.asarray(
        calibration_probabilities_v5,
        dtype=np.float64,
    )
)


def convert_to_calibration_percentile_v5(
    probabilities,
    reference_scores,
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    reference_scores = np.asarray(
        reference_scores,
        dtype=np.float64,
    )

    percentile_positions = np.searchsorted(
        reference_scores,
        probabilities,
        side="right",
    )

    return (
        percentile_positions
        / max(
            len(reference_scores),
            1,
        )
    )


test_global_percentile_v5 = (
    convert_to_calibration_percentile_v5(
        probabilities=(
            test_probabilities_v5
        ),
        reference_scores=(
            calibration_score_reference_v5
        ),
    )
)

test_global_signal_level_v5 = np.where(
    test_global_percentile_v5 >= 0.99,
    "STRONG_GLOBAL_SIGNAL",
    "NO_STRONG_GLOBAL_SIGNAL",
)

global_percentile_diagnostic_v5 = (
    pd.DataFrame({
        "target": (
            np.asarray(
                y_test_v5,
                dtype=np.int8,
            )
        ),
        "global_probability": (
            test_probabilities_v5
        ),
        "global_percentile": (
            test_global_percentile_v5
        ),
        "global_signal_level": (
            test_global_signal_level_v5
        ),
    })
)

global_signal_summary_v5 = (
    global_percentile_diagnostic_v5
    .groupby(
        "global_signal_level",
        as_index=False,
    )
    .agg(
        transaction_count=(
            "target",
            "size",
        ),
        synthetic_fraud_count=(
            "target",
            "sum",
        ),
        synthetic_fraud_rate=(
            "target",
            "mean",
        ),
        mean_probability=(
            "global_probability",
            "mean",
        ),
        minimum_percentile=(
            "global_percentile",
            "min",
        ),
        maximum_percentile=(
            "global_percentile",
            "max",
        ),
    )
    .sort_values(
        "minimum_percentile",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    global_signal_summary_v5
)

In [ ]:
# ============================================================
# 11.0 CANDIDATE V5 FEATURE IMPORTANCE
# ============================================================

feature_importance_v5 = pd.DataFrame({
    "feature": (
        lightgbm_v5_model
        .booster_
        .feature_name()
    ),
    "importance_gain": (
        lightgbm_v5_model
        .booster_
        .feature_importance(
            importance_type="gain"
        )
    ),
    "importance_split": (
        lightgbm_v5_model
        .booster_
        .feature_importance(
            importance_type="split"
        )
    ),
})

total_gain_v5 = float(
    feature_importance_v5[
        "importance_gain"
    ].sum()
)

feature_importance_v5[
    "gain_percentage"
] = np.where(
    total_gain_v5 > 0,
    (
        feature_importance_v5[
            "importance_gain"
        ]
        / total_gain_v5
        * 100
    ),
    0,
)

feature_importance_v5 = (
    feature_importance_v5
    .sort_values(
        "importance_gain",
        ascending=False,
    )
    .reset_index(drop=True)
)

used_feature_count_v5 = int(
    (
        feature_importance_v5[
            "importance_split"
        ] > 0
    ).sum()
)

zero_importance_count_v5 = int(
    (
        feature_importance_v5[
            "importance_split"
        ] == 0
    ).sum()
)

top_1_gain_share_v5 = float(
    feature_importance_v5[
        "gain_percentage"
    ].head(1).sum()
)

top_5_gain_share_v5 = float(
    feature_importance_v5[
        "gain_percentage"
    ].head(5).sum()
)

top_10_gain_share_v5 = float(
    feature_importance_v5[
        "gain_percentage"
    ].head(10).sum()
)

display(
    feature_importance_v5.head(25)
)

print(
    "Fitur digunakan:",
    used_feature_count_v5,
)

print(
    "Fitur importance nol:",
    zero_importance_count_v5,
)

print(
    "Top 1 gain share:",
    top_1_gain_share_v5,
)

print(
    "Top 5 gain share:",
    top_5_gain_share_v5,
)

print(
    "Top 10 gain share:",
    top_10_gain_share_v5,
)

In [ ]:
# ============================================================
# 11.1 CANDIDATE V5 TEMPORAL STABILITY DIAGNOSTIC
# ============================================================

def build_temporal_diagnostic_v5(
    split_name,
    source_dataframe,
    targets,
    probabilities,
    threshold,
):
    diagnostic_dataframe = pd.DataFrame({
        "step": (
            source_dataframe[
                "step"
            ]
            .to_numpy()
        ),
        "target": np.asarray(
            targets,
            dtype=np.int8,
        ),
        "probability": np.asarray(
            probabilities,
            dtype=np.float64,
        ),
    })

    diagnostic_dataframe[
        "prediction"
    ] = (
        diagnostic_dataframe[
            "probability"
        ] >= threshold
    ).astype(np.int8)

    rows = []

    for step_value, group in (
        diagnostic_dataframe
        .groupby(
            "step",
            sort=True,
        )
    ):
        y_step = (
            group["target"]
            .to_numpy()
        )

        probability_step = (
            group["probability"]
            .to_numpy()
        )

        prediction_step = (
            group["prediction"]
            .to_numpy()
        )

        if np.unique(y_step).size >= 2:
            step_pr_auc = float(
                average_precision_score(
                    y_step,
                    probability_step,
                )
            )
        else:
            step_pr_auc = np.nan

        tn, fp, fn, tp = confusion_matrix(
            y_step,
            prediction_step,
            labels=[0, 1],
        ).ravel()

        rows.append({
            "split": split_name,
            "step": int(step_value),
            "rows": int(
                len(group)
            ),
            "fraud_rate": float(
                y_step.mean()
            ),
            "mean_probability": float(
                probability_step.mean()
            ),
            "pr_auc": step_pr_auc,
            "alert_rate": float(
                prediction_step.mean()
            ),
            "precision": float(
                tp / max(tp + fp, 1)
            ),
            "recall": float(
                tp / max(tp + fn, 1)
            ),
            "false_positive_rate": float(
                fp / max(fp + tn, 1)
            ),
        })

    return pd.DataFrame(rows)


calibration_temporal_diagnostic_v5 = (
    build_temporal_diagnostic_v5(
        split_name="calibration",
        source_dataframe=(
            calibration_v3_df
        ),
        targets=y_calibration_v5,
        probabilities=(
            calibration_probabilities_v5
        ),
        threshold=(
            selected_threshold_v5
        ),
    )
)

test_temporal_diagnostic_v5 = (
    build_temporal_diagnostic_v5(
        split_name="test",
        source_dataframe=test_v3_df,
        targets=y_test_v5,
        probabilities=(
            test_probabilities_v5
        ),
        threshold=(
            selected_threshold_v5
        ),
    )
)

temporal_diagnostic_v5 = pd.concat(
    [
        calibration_temporal_diagnostic_v5,
        test_temporal_diagnostic_v5,
    ],
    axis=0,
    ignore_index=True,
)

display(
    temporal_diagnostic_v5
)

In [ ]:
# ============================================================
# 11.2 CANDIDATE V5 ALERT RATE CONSISTENCY CHECK
# ============================================================

summary_alert_rate_v5 = float(
    test_summary_v5
    .iloc[0]["alert_rate"]
)

prediction_alert_rate_v5 = float(
    np.asarray(
        test_predictions_v5,
        dtype=np.int8,
    ).mean()
)

weighted_temporal_alert_rate_v5 = float(
    np.average(
        test_temporal_diagnostic_v5[
            "alert_rate"
        ],
        weights=(
            test_temporal_diagnostic_v5[
                "rows"
            ]
        ),
    )
)

summary_vs_prediction_difference_v5 = abs(
    summary_alert_rate_v5
    - prediction_alert_rate_v5
)

summary_vs_temporal_difference_v5 = abs(
    summary_alert_rate_v5
    - weighted_temporal_alert_rate_v5
)

prediction_vs_temporal_difference_v5 = abs(
    prediction_alert_rate_v5
    - weighted_temporal_alert_rate_v5
)

alert_rate_consistency_v5 = pd.DataFrame([
    {
        "source": (
            "test_summary_v5"
        ),
        "alert_rate": (
            summary_alert_rate_v5
        ),
    },
    {
        "source": (
            "test_predictions_v5.mean"
        ),
        "alert_rate": (
            prediction_alert_rate_v5
        ),
    },
    {
        "source": (
            "weighted_test_temporal_diagnostic"
        ),
        "alert_rate": (
            weighted_temporal_alert_rate_v5
        ),
    },
])

display(
    alert_rate_consistency_v5
)

print(
    "Difference summary vs prediction:",
    summary_vs_prediction_difference_v5,
)

print(
    "Difference summary vs temporal:",
    summary_vs_temporal_difference_v5,
)

print(
    "Difference prediction vs temporal:",
    prediction_vs_temporal_difference_v5,
)

is_summary_prediction_consistent_v5 = (
    np.isclose(
        summary_alert_rate_v5,
        prediction_alert_rate_v5,
        atol=1e-12,
        rtol=0.0,
    )
)

is_summary_temporal_consistent_v5 = (
    np.isclose(
        summary_alert_rate_v5,
        weighted_temporal_alert_rate_v5,
        atol=1e-12,
        rtol=0.0,
    )
)

is_prediction_temporal_consistent_v5 = (
    np.isclose(
        prediction_alert_rate_v5,
        weighted_temporal_alert_rate_v5,
        atol=1e-12,
        rtol=0.0,
    )
)

all_alert_rates_consistent_v5 = (
    is_summary_prediction_consistent_v5
    and is_summary_temporal_consistent_v5
    and is_prediction_temporal_consistent_v5
)

print(
    "Summary dan prediction konsisten:",
    is_summary_prediction_consistent_v5,
)

print(
    "Summary dan temporal konsisten:",
    is_summary_temporal_consistent_v5,
)

print(
    "Prediction dan temporal konsisten:",
    is_prediction_temporal_consistent_v5,
)

print(
    "Semua alert rate konsisten:",
    all_alert_rates_consistent_v5,
)

if not all_alert_rates_consistent_v5:
    print(
        "\nWARNING:"
        "\nVariabel berasal dari run atau model yang berbeda."
        "\nRestart session dan jalankan ulang cell 08.3 sampai 11.2 "
        "secara berurutan."
    )
else:
    print(
        "\nAlert rate konsisten. "
        "Artefak aman untuk diekspor dari run ini."
    )

In [ ]:
# ============================================================
# 12.0 EXPORT GLOBAL RANKER V5 ARTIFACTS
# ============================================================

from pathlib import Path
from datetime import datetime, timezone


def to_json_compatible_v5(value):
    """Mengubah scalar/collection NumPy dan pandas menjadi JSON-safe."""
    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, np.ndarray):
        return [
            to_json_compatible_v5(item)
            for item in value.tolist()
        ]

    if isinstance(value, dict):
        return {
            str(key): to_json_compatible_v5(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_json_compatible_v5(item)
            for item in value
        ]

    if pd.isna(value):
        return None

    return value


GLOBAL_RANKER_NAME = (
    "global_ranker_v5_momtsim_v2"
)

GLOBAL_RANKER_VERSION = "5.0.0"

GLOBAL_RANKER_DIR = (
    Path(MODEL_DIR)
    / GLOBAL_RANKER_NAME
)

GLOBAL_RANKER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Export directory:", GLOBAL_RANKER_DIR)

# ============================================================
# FILE PATHS
# ============================================================

MODEL_PATH_V5 = (
    GLOBAL_RANKER_DIR
    / "global_ranker_v5.joblib"
)

FEATURE_CONTRACT_PATH_V5 = (
    GLOBAL_RANKER_DIR
    / "feature_contract_v5.json"
)

METADATA_PATH_V5 = (
    GLOBAL_RANKER_DIR
    / "model_metadata_v5.json"
)

CALIBRATION_REFERENCE_PATH_V5 = (
    GLOBAL_RANKER_DIR
    / "calibration_score_reference_v5.npy"
)

CALIBRATION_THRESHOLD_PATH_V5 = (
    GLOBAL_RANKER_DIR
    / "calibration_threshold_v5.json"
)

TOP_FRACTION_THRESHOLDS_PATH_V5 = (
    GLOBAL_RANKER_DIR
    / "top_fraction_thresholds_v5.json"
)

EVALUATION_METRICS_PATH_V5 = (
    GLOBAL_RANKER_DIR
    / "evaluation_metrics_v5.json"
)

FEATURE_IMPORTANCE_PATH_V5 = (
    GLOBAL_RANKER_DIR
    / "feature_importance_v5.csv"
)

BENCHMARK_RESULTS_PATH_V5 = (
    GLOBAL_RANKER_DIR
    / "benchmark_results_v5.csv"
)

FEATURE_DIAGNOSTIC_PATH_V5 = (
    GLOBAL_RANKER_DIR
    / "feature_diagnostic_v5.csv"
)

TOP_K_METRICS_PATH_V5 = (
    GLOBAL_RANKER_DIR
    / "top_k_metrics_v5.csv"
)

# ============================================================
# SAVE LIGHTGBM MODEL
# ============================================================

joblib.dump(
    lightgbm_v5_model,
    MODEL_PATH_V5,
)

# ============================================================
# SAVE FEATURE CONTRACT
# ============================================================

feature_contract_v5 = {
    "feature_contract_version": "5.0.0",
    "base_feature_version": "candidate_v3",
    "interaction_feature_version": (
        "candidate_v4_interactions_reused"
    ),
    "dataset_version": "momtsim_v2",
    "model_scope": "TRANSFER",
    "numeric_features": list(
        V5_NUMERIC_FEATURES
    ),
    "base_features": list(
        V5_BASE_FEATURES
    ),
    "derived_features": list(
        V5_DERIVED_FEATURES
    ),
    "removed_constant_features": list(
        constant_features_v5
    ),
    "feature_count": int(
        len(V5_NUMERIC_FEATURES)
    ),
    "required_audit_columns": [
        "step",
        "initiator",
        "recipient",
        "transaction_type",
    ],
    "target_column": "target",
    "unsupported_production_features": [
        "old_bal_recipient",
        "new_bal_recipient",
        "recipient_balance_change",
        "recipient_balance_error",
        "amount_to_recipient_balance_ratio",
        "recipient_old_balance_zero",
    ],
}

with open(
    FEATURE_CONTRACT_PATH_V5,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        to_json_compatible_v5(feature_contract_v5),
        file,
        indent=2,
        ensure_ascii=False,
    )

# ============================================================
# SAVE CALIBRATION SCORE REFERENCE
# ============================================================

calibration_score_reference_v5 = np.sort(
    np.asarray(
        calibration_probabilities_v5,
        dtype=np.float64,
    )
)

np.save(
    CALIBRATION_REFERENCE_PATH_V5,
    calibration_score_reference_v5,
)

# ============================================================
# SAVE CALIBRATION THRESHOLD
# Threshold ini hasil eksperimen global classifier.
# Jangan langsung dipakai sebagai threshold produk.
# ============================================================

calibration_threshold_config_v5 = {
    "threshold_version": "5.0.0",
    "threshold": float(
        selected_threshold_v5
    ),
    "selection_split": "calibration",
    "selection_strategy": str(
        calibration_summary_v5.iloc[0][
            "selection_strategy"
        ]
    ),
    "recommended_product_usage": False,
    "usage_note": (
        "Threshold eksperimen global classifier pada MoMTSim V2. "
        "Global ranker V5 direkomendasikan sebagai ranking signal "
        "dan input score fusion."
    ),
}

with open(
    CALIBRATION_THRESHOLD_PATH_V5,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        to_json_compatible_v5(
            calibration_threshold_config_v5
        ),
        file,
        indent=2,
        ensure_ascii=False,
    )

# ============================================================
# SAVE CALIBRATION-LOCKED TOP-FRACTION THRESHOLDS
# ============================================================

top_fraction_thresholds_v5 = {}

for configured_fraction in GLOBAL_TOP_FRACTIONS_V5:
    threshold_value = float(
        np.quantile(
            calibration_probabilities_v5,
            1.0 - configured_fraction,
        )
    )

    threshold_key = (
        f"top_{int(configured_fraction * 100)}"
        "_percent"
    )

    top_fraction_thresholds_v5[
        threshold_key
    ] = {
        "fraction": float(configured_fraction),
        "percent": float(
            configured_fraction * 100
        ),
        "threshold": threshold_value,
        "reference_split": "calibration",
    }

with open(
    TOP_FRACTION_THRESHOLDS_PATH_V5,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        to_json_compatible_v5(
            top_fraction_thresholds_v5
        ),
        file,
        indent=2,
        ensure_ascii=False,
    )

# ============================================================
# SAVE TEST EVALUATION
# ============================================================

test_metrics_v5 = (
    test_summary_v5
    .iloc[0]
    .to_dict()
)

evaluation_metrics_v5 = {
    key: to_json_compatible_v5(value)
    for key, value in test_metrics_v5.items()
}

evaluation_metrics_v5[
    "pr_auc_lift"
] = float(
    evaluation_metrics_v5["pr_auc"]
    / max(
        evaluation_metrics_v5["fraud_rate"],
        1e-12,
    )
)

evaluation_metrics_v5[
    "top_k_metrics"
] = to_json_compatible_v5(
    top_k_metrics_v5.to_dict(
        orient="records"
    )
)

evaluation_metrics_v5[
    "baseline_candidate_v4_momtsim_v1"
] = to_json_compatible_v5(
    V4_V1_BASELINE_TEST_METRICS
)

evaluation_metrics_v5[
    "class_prior_baseline"
] = to_json_compatible_v5(
    class_prior_baseline_v5.to_dict(
        orient="records"
    )
)

evaluation_metrics_v5[
    "cold_start_segment_metrics"
] = to_json_compatible_v5(
    cold_start_segment_metrics_v5.to_dict(
        orient="records"
    )
)

evaluation_metrics_v5[
    "global_signal_policy"
] = {
    "strong_signal_min_percentile": 0.99,
    "otherwise": "NO_STRONG_GLOBAL_SIGNAL",
}

with open(
    EVALUATION_METRICS_PATH_V5,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        to_json_compatible_v5(
            evaluation_metrics_v5
        ),
        file,
        indent=2,
        ensure_ascii=False,
    )

# ============================================================
# SAVE MODEL METADATA
# ============================================================

model_metadata_v5 = {
    "model_name": GLOBAL_RANKER_NAME,
    "model_version": GLOBAL_RANKER_VERSION,
    "experiment_version": EXPERIMENT_VERSION,
    "dataset_version": DATASET_VERSION,
    "previous_model": PREVIOUS_GLOBAL_MODEL,
    "model_type": "LightGBM",
    "model_role": "global_transaction_risk_ranker",
    "decision_scope": "TRANSFER",
    "label_column": "is_fraud",
    "label_meaning": (
        "Synthetic fraud generated by MoMTSim. "
        "Not a gambling label."
    ),
    "feature_contract_version": "5.0.0",
    "feature_count": int(
        len(V5_NUMERIC_FEATURES)
    ),
    "selected_configuration": (
        best_config_name_v5
    ),
    "best_iteration": int(
        selected_best_iteration_v5
    ),
    "calibration_threshold": float(
        selected_threshold_v5
    ),
    "strong_global_signal_percentile": 0.99,
    "recommended_usage": [
        "global probability",
        "calibration percentile",
        "top-one-percent extreme-risk ranking signal",
        "input for Model 1 score fusion",
    ],
    "not_recommended_usage": [
        "standalone fraud decision",
        "standalone gambling detection",
        "standalone user alert",
    ],
    "created_at_utc": (
        datetime.now(timezone.utc)
        .isoformat()
    ),
}

with open(
    METADATA_PATH_V5,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        to_json_compatible_v5(
            model_metadata_v5
        ),
        file,
        indent=2,
        ensure_ascii=False,
    )

# ============================================================
# SAVE DIAGNOSTIC TABLES
# ============================================================

feature_importance_v5.to_csv(
    FEATURE_IMPORTANCE_PATH_V5,
    index=False,
)

benchmark_results_v5.to_csv(
    BENCHMARK_RESULTS_PATH_V5,
    index=False,
)

feature_diagnostic_v5.to_csv(
    FEATURE_DIAGNOSTIC_PATH_V5,
    index=False,
)

top_k_metrics_v5.to_csv(
    TOP_K_METRICS_PATH_V5,
    index=False,
)

print("Global ranker V5 berhasil diekspor.")
print("Model:", MODEL_PATH_V5)
print("Feature contract:", FEATURE_CONTRACT_PATH_V5)
print("Metadata:", METADATA_PATH_V5)
print("Calibration reference:", CALIBRATION_REFERENCE_PATH_V5)
print("Top-fraction thresholds:", TOP_FRACTION_THRESHOLDS_PATH_V5)


In [ ]:
# ============================================================
# 12.1 VERIFY EXPORTED GLOBAL RANKER V5
# ============================================================

required_artifact_paths_v5 = [
    MODEL_PATH_V5,
    FEATURE_CONTRACT_PATH_V5,
    METADATA_PATH_V5,
    CALIBRATION_REFERENCE_PATH_V5,
    CALIBRATION_THRESHOLD_PATH_V5,
    TOP_FRACTION_THRESHOLDS_PATH_V5,
    EVALUATION_METRICS_PATH_V5,
    FEATURE_IMPORTANCE_PATH_V5,
    BENCHMARK_RESULTS_PATH_V5,
    FEATURE_DIAGNOSTIC_PATH_V5,
    TOP_K_METRICS_PATH_V5,
]

missing_artifacts_v5 = [
    str(path)
    for path in required_artifact_paths_v5
    if not path.exists()
]

if missing_artifacts_v5:
    raise FileNotFoundError(
        "Artefak V5 tidak ditemukan:\n"
        + "\n".join(
            missing_artifacts_v5
        )
    )

loaded_global_ranker_v5 = (
    joblib.load(
        MODEL_PATH_V5
    )
)

with open(
    FEATURE_CONTRACT_PATH_V5,
    "r",
    encoding="utf-8",
) as file:
    loaded_feature_contract_v5 = (
        json.load(file)
    )

with open(
    METADATA_PATH_V5,
    "r",
    encoding="utf-8",
) as file:
    loaded_metadata_v5 = (
        json.load(file)
    )

loaded_calibration_reference_v5 = (
    np.load(
        CALIBRATION_REFERENCE_PATH_V5
    )
)

if (
    loaded_feature_contract_v5[
        "numeric_features"
    ]
    != list(V5_NUMERIC_FEATURES)
):
    raise ValueError(
        "Feature contract hasil export "
        "tidak sama dengan fitur training."
    )

if (
    loaded_global_ranker_v5
    .n_features_in_
    != len(V5_NUMERIC_FEATURES)
):
    raise ValueError(
        "Jumlah fitur pada model export "
        "tidak sesuai feature contract."
    )

if (
    loaded_calibration_reference_v5.ndim
    != 1
):
    raise ValueError(
        "Calibration reference harus "
        "berupa array satu dimensi."
    )

verification_probabilities_v5 = (
    loaded_global_ranker_v5
    .predict_proba(
        X_test_v5.head(20)
    )[:, 1]
)

original_probabilities_sample_v5 = (
    test_probabilities_v5[:20]
)

maximum_probability_difference_v5 = float(
    np.max(
        np.abs(
            verification_probabilities_v5
            - original_probabilities_sample_v5
        )
    )
)

if (
    maximum_probability_difference_v5
    > 1e-10
):
    raise ValueError(
        "Prediksi model setelah reload "
        "berbeda dari model asli."
    )

artifact_verification_v5 = pd.DataFrame([
    {
        "artifact": (
            path.name
        ),
        "exists": (
            path.exists()
        ),
        "size_bytes": (
            path.stat().st_size
        ),
    }
    for path in required_artifact_paths_v5
])

display(
    artifact_verification_v5
)

print(
    "Model name:",
    loaded_metadata_v5[
        "model_name"
    ],
)

print(
    "Feature count:",
    loaded_feature_contract_v5[
        "feature_count"
    ],
)

print(
    "Calibration reference rows:",
    len(
        loaded_calibration_reference_v5
    ),
)

print(
    "Maximum reload prediction difference:",
    maximum_probability_difference_v5,
)

print(
    "Verifikasi artefak V5 berhasil."
)

In [ ]:
# ============================================================
# 12.2 GLOBAL RANKER V5 INFERENCE INTERFACE
# ============================================================

def convert_score_to_percentile_v5(
    probability,
    calibration_reference,
):
    calibration_reference = np.asarray(
        calibration_reference,
        dtype=np.float64,
    )

    position = np.searchsorted(
        calibration_reference,
        float(probability),
        side="right",
    )

    percentile = (
        position
        / max(
            len(calibration_reference),
            1,
        )
    )

    return float(
        np.clip(
            percentile,
            0.0,
            1.0,
        )
    )


def map_global_signal_level_v5(
    percentile,
):
    if percentile >= 0.99:
        return "STRONG_GLOBAL_SIGNAL"

    return "NO_STRONG_GLOBAL_SIGNAL"


def predict_global_ranker_v5(
    feature_row,
    model,
    feature_contract,
    calibration_reference,
):
    expected_features = (
        feature_contract[
            "numeric_features"
        ]
    )

    missing_features = [
        feature
        for feature in expected_features
        if feature not in feature_row.columns
    ]

    if missing_features:
        raise ValueError(
            "Fitur inference tidak lengkap: "
            + str(
                missing_features
            )
        )

    model_matrix = (
        feature_row[
            expected_features
        ]
        .astype("float32")
    )

    probability = float(
        model.predict_proba(
            model_matrix
        )[0, 1]
    )

    percentile = (
        convert_score_to_percentile_v5(
            probability=probability,
            calibration_reference=(
                calibration_reference
            ),
        )
    )

    signal_level = (
        map_global_signal_level_v5(
            percentile
        )
    )

    return {
        "global_probability": probability,
        "global_percentile": percentile,
        "global_signal_level": (
            signal_level
        ),
        "model_name": (
            GLOBAL_RANKER_NAME
        ),
        "model_version": (
            GLOBAL_RANKER_VERSION
        ),
    }


global_inference_example_v5 = (
    predict_global_ranker_v5(
        feature_row=(
            X_test_v5.iloc[
                [0]
            ]
        ),
        model=(
            loaded_global_ranker_v5
        ),
        feature_contract=(
            loaded_feature_contract_v5
        ),
        calibration_reference=(
            loaded_calibration_reference_v5
        ),
    )
)

print(
    json.dumps(
        global_inference_example_v5,
        indent=2,
        ensure_ascii=False,
    )
)